In [ ]:
import os
from typing import Dict, List, Literal, Tuple, Optional
import logging
import math
import pickle

import json
from datetime import datetime
from pathlib import Path

import category_encoders as ce
from catboost import CatBoostRegressor
import kan
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from pycatch22 import catch22_all
from scipy.interpolate import PchipInterpolator
from scipy.special import softmax

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

import xarray as xr
import zarr
import shutil
from glob import glob

from forecasting_module import TimeGPTForecaster, SARIMAXForecaster
from utils import ForecastUtils, make_sample_splits, read_params

from lstm_network import LSTMModel, LSTMTrainer

import other_encoders.autoencoders as ae
import other_encoders.train_autoencoders as train_ae
from other_encoders.ts2vec_encoder import TS2VecEncoder
from other_encoders.custom_stuff import Decoder, MLPHead, ProjectionHead, TorchWrapper

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
interim_data_loc = "../interim_data"
encoders_folder  = "other_encoders"

params          = read_params("params2.yaml")
desired_dataset = params["basics"]["desired_dataset"]


In [ ]:
"""ECG dataset"""
if desired_dataset == "ecg":
    from dataset_loaders import ECGLoader

    ECG_data_path = "../public_datasets/3D/ptb-xl-1.0.3"
    loader        = ECGLoader(ECG_data_path)
    X_ecg, y_ecg, sampling_rate, leads = loader.load_dataset(sampling="lr", target="diagnostic_superclass_multi",
                                                            segment_duration_sec=10.0, max_records=100, continuous_target=True)
    print(f"X shape: {X_ecg.shape}, X size (MB): {X_ecg.nbytes / 1024**2:.2f}")   # (patients, time, leads)
    print(f"y shape: {y_ecg.shape}, y size (MB): {y_ecg.nbytes / 1024**2:.4f}")  # (patients, conditions)


In [ ]:
"""Argoverse dataset"""

def process_argoverse_parquet(scenario_parquet_path: str):
    """Convert Argoverse Parquet scenario to X (3D) and y (2D)
    Follows these rules:
        1. track_id != focal_track_id AND observed = true → include in X
        2. track_id  = focal_track_id AND observed = true → include in X
        3. track_id  = focal_track_id AND observed = false → include in y only
        4. track_id != focal_track_id AND observed = false → ignore"""
    df       = pd.read_parquet(scenario_parquet_path)
    focal_id = df['focal_track_id'].iloc[0]

    # Remove irrelevant rows (rule 4), but keep focal agent even if unobserved (rule 3)
    df = df[(df['observed'] == True) | (df['track_id'] == focal_id)]

    agent_past_list   = []
    focal_future_list = []

    for track_id, track_data in df.groupby('track_id'):
        xy_pos     = track_data[['position_x', 'position_y']].values
        headings   = track_data['heading'].values
        velocities = track_data[['velocity_x', 'velocity_y']].values
        is_observed= track_data['observed'].values

        agent_past   = []
        focal_future = []

        for t, obs in enumerate(is_observed):
            if obs:  # observed = True → goes into X
                agent_past.append([xy_pos[t][0], xy_pos[t][1],
                                   headings[t], velocities[t][0],
                                   velocities[t][1]])
            elif track_id == focal_id:  # observed = False and focal → goes into y
                focal_future.append(xy_pos[t])  # only x, y
        if agent_past:
            agent_past_list.append(np.array(agent_past))
        if focal_future and track_id == focal_id:
            focal_future_list = np.array(focal_future)

    # pad sequences to longest agent
    max_agent_len   = max(len(p) for p in agent_past_list)
    X_agents_padded = np.array([np.pad(p, ((0, max_agent_len-len(p)), (0,0)), 'constant') for p in agent_past_list])

    return X_agents_padded, np.array(focal_future_list)

if desired_dataset == "argoverse":
    argoverse_data_path = "../public_datasets/3D/argoverse_forecasting"
    folder_name    = "00a0ec58-1fb9-4a2b-bfd7-f4e5da7a9eff"
    file_name      = "scenario_00a0ec58-1fb9-4a2b-bfd7-f4e5da7a9eff.parquet"
    X_argo, y_argo = process_argoverse_parquet(f"{argoverse_data_path}/{folder_name}/{file_name}")
    print(X_argo.shape, y_argo.shape)


In [ ]:
"""Weather dataset"""

class WeatherDataset:
    """Loads and preprocesses WeatherBench-style datasets for ML.
    Supports multiple output shapes for X and y"""
    def __init__(self, dataset_folder: str, variables_X: list, variables_y: list, freq="6H"):
        self.dataset_folder = dataset_folder
        self.variables_X    = variables_X
        self.variables_y    = variables_y
        self.freq   = freq
        self.X_xarr = None
        self.y_xarr = None

    @staticmethod
    def open_zarr_variable(folder_path: str, varname: str) -> xr.DataArray:
        """Open a folder containing a single variable as xarray.DataArray"""
        arr = zarr.open(folder_path, mode="r")
        time= pd.date_range("1959-01-01", periods=arr.shape[0], freq="6H")
        lat = np.linspace(-90, 90, arr.shape[1])
        lon = np.linspace(0, 360, arr.shape[2], endpoint=False)
        return xr.DataArray(arr, dims=["time", "lat", "lon"], coords={"time": time, "lat": lat, "lon": lon}, name=varname)

    def load_dataset(self):
        """Load all variables into xarray Datasets for X and y"""
        X_data      = {var: self.open_zarr_variable(f"{self.dataset_folder}/{var}", var) for var in self.variables_X}
        y_data      = {var: self.open_zarr_variable(f"{self.dataset_folder}/{var}", var) for var in self.variables_y}
        self.X_xarr = xr.Dataset(X_data)
        self.y_xarr = xr.Dataset(y_data)
        return self.X_xarr, self.y_xarr

    @staticmethod
    def prepare_X_features(X_ds: xr.Dataset, mode: str = "autoencoder", last_timesteps: int = 1) -> np.ndarray:
        """Convert xarray.Dataset of features to NumPy array
        - "autoencoder" [for autoencoder]  -> keep channels and spatial dims (time x channels x lat x lon)
        - "3d" [for 3D CNN]                -> keep channels, flatten spatial dims (time x channels x lat*lon)
        - "lstm" [for LSTM/Transformer]    -> flatten spatial dims, keep time (time x features)
        - "catboost" [for CatBoost]        -> flatten channels and spatial dims for last timestep only (1 x features)"""
        X_arr = np.stack([X_ds[var].values for var in X_ds.data_vars], axis=1)  # (time, channels, lat, lon)
        if mode == "autoencoder":
            return X_arr
        elif mode == "3d":
            return X_arr.reshape(X_arr.shape[0], X_arr.shape[1], -1)  # (time, channels, lat*lon)
        elif mode == "lstm":
            time_dim, channels, lat, lon = X_arr.shape
            return X_arr.reshape(time_dim, channels * lat * lon)
        elif mode == "catboost":
            X_last = X_arr[-last_timesteps:]
            return X_last.reshape(last_timesteps, -1)
        else:
            raise ValueError("mode must be one of ['autoencoder','3d','lstm','catboost']")

    @staticmethod
    def prepare_y_targets(y_ds: xr.Dataset, mode: str = "3d") -> np.ndarray:
        """Convert xarray.Dataset of targets to NumPy array"""
        y_arr  = np.stack([y_ds[var].values for var in y_ds.data_vars], axis=0)  # (vars, time, lat, lon)
        y_last = y_arr[:, -1, :, :]  # take last timestep
        if mode == "3d":
            return y_last  # (vars, lat, lon)
        elif mode == "flatten":
            return y_last.reshape(1, -1)  # (1, vars*lat*lon)
        elif mode == "collapse":
            return y_last.mean(axis=1)  # (vars, lon)
        else:
            raise ValueError("mode must be one of ['3d','flatten','collapse']")

    @staticmethod
    def reshape_for_ml(X: np.ndarray) -> np.ndarray:
        """Convert X from (time, channels, lat*lon) -> (lat*lon, time, channels)
        for per-grid-cell sequence models like timeVAE."""
        return np.moveaxis(X, [0, 1, 2], [1, 2, 0])

    @staticmethod
    def flatten_y(y: np.ndarray) -> np.ndarray:
        """Flatten y from (vars, lat, lon) -> (lat*lon, vars)
        to align with reshaped X."""
        return y.reshape(y.shape[0], -1).T


if desired_dataset == "weather":
    dataset_folder = "../public_datasets/3D/weather_bench"
    variables_X    = ["2m_temperature", "10m_u_component_of_wind", "10m_v_component_of_wind"]
    variables_y    = ["mean_sea_level_pressure", "total_precipitation_6hr"]

    weather        = WeatherDataset(dataset_folder, variables_X, variables_y)
    X_xarr, y_xarr = weather.load_dataset()

    # Prepare X
    X_3d = weather.prepare_X_features(X_xarr, mode="3d") # (time, channels, lat*lon)
    X_3d = weather.reshape_for_ml(X_3d)                  # (lat*lon, time, channels)

    # Prepare y
    y_3d = weather.prepare_y_targets(y_xarr, mode="3d")  # (vars, lat, lon)
    y_2d = weather.flatten_y(y_3d)                       # (lat*lon, vars)

    # Ready for ML
    X_weather, y_weather = X_3d, y_2d
    print(X_weather.shape, y_weather.shape)  # (2048, 92044, 3), (2048, 2)
    print(f"X_weather memory: {X_weather.nbytes/1024**2:.2f} MB")


In [ ]:
"Camels DE"

def load_X_from_scratch(timeseries_folder, zarr_path):
    # --- Delete Zarr if needed ---
    if os.path.exists(zarr_path):
        shutil.rmtree(zarr_path)

    # --- Load X ---
    ts_files  = sorted(glob(os.path.join(timeseries_folder, "*.csv")))
    ts_arrays = []

    for f in ts_files:
        df  = pd.read_csv(f, index_col=0)  # (time, catchments)
        df  = df.drop(columns=['date', 'discharge_vol_obs', 'discharge_spec_obs', 'water_level_obs'])
        arr = df.values
        ts_arrays.append(arr)

    # Stack along new axis -> (time, catchments, variables)
    X_np = np.stack(ts_arrays, axis=2)
    X_np = np.transpose(X_np, (2, 0, 1))
    print("X shape after transpose:", X_np.shape)  # (1582, 25568, 21)

    X_da = xr.DataArray(X_np, dims=("catchment", "time", "variable"))
    X_da.to_dataset(name="X").to_zarr(zarr_path, mode="w")
    print("Saved X to Zarr:", zarr_path)

    X_da_lazy = xr.open_zarr(zarr_path)["X"].values
    return X_da_lazy

def load_y_from_scratch(attributes_folder):
    """Also removed str cols"""
    attr_files = glob(os.path.join(attributes_folder, "CAMELS_DE_*.csv"))
    y_list     = [pd.read_csv(f, index_col=0) for f in attr_files]
    y_germany  = pd.concat(y_list, axis=1)
    y_germany = y_germany.select_dtypes(exclude='object').to_numpy()
    return y_germany

if desired_dataset == "germany_catchment":
    camels_root       = "../public_datasets/3D/camels_de"
    attributes_folder = camels_root
    timeseries_folder = os.path.join(camels_root, "timeseries")
    zarr_path         = os.path.join(camels_root, "camels_de_timeseries.zarr")

    if os.path.exists(zarr_path):
        X_germany = xr.open_zarr(zarr_path)["X"].values
    else:
        X_germany = load_X_from_scratch(timeseries_folder, zarr_path)
    y_germany = load_y_from_scratch(attributes_folder)

    print(f"X shape: {X_germany.shape} ({X_germany.nbytes / 1024**2:.2f} MB)")  # (1582, 25568, 21)
    print(f"y shape: {y_germany.shape} ({y_germany.nbytes / 1024**2:.2f} MB)")  # (1582, 25568, 21)


In [ ]:
"""India dataset"""

if desired_dataset == "india_catchment":
    data_path      = "../public_datasets/3D/india_catchments"
    forcing_folder = "catchment_mean_forcings"
    clim_file      = "attributes_csv/camels_ind_clim.csv"

    # Load y (climate attributes)
    y_df    = pd.read_csv(f"{data_path}/{clim_file}")
    catchment_ids = y_df.iloc[:,0].astype(int).values
    y_india = y_df.iloc[:, 1:]

    # Load X (forcing time series)
    forcing_files    = sorted(os.listdir(f"{data_path}/{forcing_folder}"))
    X_list, file_ids = [], []

    for f in forcing_files:
        path = os.path.join(f"{data_path}/{forcing_folder}", f)
        df   = pd.read_csv(path)
        X_list.append(df.drop(columns=['year','month','day','pet(mm/day)']).values)
        file_ids.append(int(f.split('.')[0]))

    X       = np.stack(X_list, axis=0)
    order   = [file_ids.index(cid) for cid in catchment_ids]
    X_india = X[order]

    print("X_india shape:", X_india.shape, "y_india shape:", y_india.shape)


In [ ]:
"""China Weather 2k data"""

if desired_dataset == "china_weather":

    dataset_location = "../public_datasets/3D/china_weather/weather2k.npy"
    # dataset_location = "/Users/fouadabiad/Downloads/weather2k.npy"
    china_data       = np.load(dataset_location, mmap_mode='r')  # read-only memory map
    china_data       = china_data.transpose(0,2,1) # (stations, timesteps, features)

    y_indices = [4, 5, 6, 7, 10] # [T, mnt, mxt, rh, ws], to remove
    y         = china_data[:, -1, y_indices] # last timestep
    mask      = np.ones(china_data.shape[2], dtype=bool)
    mask[y_indices] = False
    X         = china_data[:, :, mask]

    print(f"X shape: {X.shape} ({X.nbytes / 1024**2:.2f} MB)")  # (patients, conditions)
    print(f"y shape: {y.shape} ({y.nbytes / 1024**2:.2f} MB)")  # (patients, conditions)


In [ ]:
"""[RUN ME] Preprocess dataset, as class"""
from dataset_loaders import ECGLoader
import category_encoders as ce

class DatasetPreprocessor:
    """Preprocess datasets: downsample, train/test split, categorical encoding, and scaling."""
    
    def __init__(self, page_frac=0.1, row_frac=0.4, test_size=0.2, scale_X=True, random_seed=42):
        self.page_frac  = page_frac
        self.row_frac   = row_frac
        self.test_size  = test_size
        self.scale_X    = scale_X
        self.random_seed= random_seed
        self.y_mean     = None
        self.y_std      = None
        self.X_scaler   = None

    def fit_transform(self, X: np.ndarray, y: np.ndarray, split_X: bool = True) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Downsample, split, encode, and scale dataset. X always stays 3D."""
        self._prepare_targets(y)
        X_small, y_small = self._downsample_pages_and_rows(X, self.y_df)

        # =======
        n_samples = X_small.shape[0]
        n_test    = int(n_samples * self.test_size)

        # 1. Create and shuffle indices based on the random seed
        rng     = np.random.default_rng(self.random_seed)
        indices = rng.permutation(n_samples)
        
        # 2. Define train and test index sets
        test_indices  = indices[:n_test]
        train_indices = indices[n_test:]

        # 3. Split X and Y simultaneously using the shared indices
        y_train_raw, y_test_raw = y_small[train_indices], y_small[test_indices]
        X_train, X_test         = X_small[train_indices], X_small[test_indices]
        y_train, y_test = self._encode_categorical(y_train_raw, y_test_raw)
        y_train_scaled, y_test_scaled = self._scale_targets(y_train, y_test)
        # =======
        # y_train, y_test  = train_test_split(y_small, test_size=self.test_size, shuffle=False)
        # y_train, y_test  = self._encode_categorical(y_train, y_test)
        # y_train_scaled, y_test_scaled = self._scale_targets(y_train, y_test)

        if not split_X:
            return X_small, y_train_scaled, y_test_scaled, None

        # Split X along first axis (pages) without flattening
        # X_train, X_test = train_test_split(X_small, test_size=self.test_size, shuffle=False)

        if self.scale_X:
            # Optionally scale while keeping 3D shape
            ns, nr, nf = X_train.shape
            ns_test, nr_test, nf_test = X_test.shape

            # Flatten temporarily for StandardScaler
            X_train_flat   = X_train.reshape(ns, -1)
            X_test_flat    = X_test.reshape(ns_test, -1)
            self.X_scaler  = StandardScaler()
            X_train_scaled = self.X_scaler.fit_transform(X_train_flat).reshape(ns, nr, nf)
            X_test_scaled  = self.X_scaler.transform(X_test_flat).reshape(ns_test, nr_test, nf_test)
            return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled
        else:
            return X_train, X_test, y_train_scaled, y_test_scaled

    def _prepare_targets(self, y):
        if isinstance(y, np.ndarray):
            self.y_df         = pd.DataFrame(y)
            self.cat_cols     = []
            self.numeric_cols = self.y_df.columns.tolist()
        else:
            self.y_df         = y.copy()
            self.numeric_cols = self.y_df.select_dtypes(include=[np.number]).columns.tolist()
            self.cat_cols     = self.y_df.select_dtypes(include=['object','category']).columns.tolist()

    def _downsample_pages_and_rows(self, X, y_df):
        n_pages, n_rows, _ = X.shape
        num_pages = max(1, int(n_pages * self.page_frac))
        num_rows  = max(1, int(n_rows * self.row_frac))
        page_idx  = np.linspace(0, n_pages-1, num_pages, dtype=int)
        row_idx   = np.linspace(0, n_rows-1, num_rows, dtype=int)
        X_small   = X[page_idx][:, row_idx, :]
        y_small   = y_df.iloc[page_idx].values if isinstance(y_df, pd.DataFrame) else y_df[page_idx]
        return X_small, y_small

    def _encode_categorical(self, y_train, y_test):
        if len(self.cat_cols) > 0:
            idx         = [self.y_df.columns.get_loc(c) for c in self.cat_cols]
            y_train_df  = pd.DataFrame(y_train[:, idx], columns=self.cat_cols)
            y_test_df   = pd.DataFrame(y_test[:, idx], columns=self.cat_cols)
            encoder     = ce.TargetEncoder(cols=self.cat_cols)
            y_train_enc = encoder.fit_transform(y_train_df, y_train[:,0])
            y_test_enc  = encoder.transform(y_test_df)
            y_train[:, idx] = y_train_enc.values
            y_test[:, idx]  = y_test_enc.values
        return y_train.astype(float), y_test.astype(float)

    def _scale_targets(self, y_train, y_test):
        self.y_mean    = y_train.mean(axis=0)
        self.y_std     = y_train.std(axis=0)
        self.y_std[self.y_std == 0] = 1.0
        y_train_scaled = (y_train - self.y_mean) / self.y_std
        y_test_scaled  = (y_test - self.y_mean) / self.y_std
        y_train_scaled = np.atleast_2d(y_train_scaled)
        y_test_scaled  = np.atleast_2d(y_test_scaled)
        return y_train_scaled, y_test_scaled

def load_and_preprocess_dataset(desired_dataset: str, split_X: bool = False, segment_duration_sec: int = 200,
                                max_records: int = 2000) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Load a public 3D dataset, downsample, preprocess, and optionally flatten/scale X."""
    
    # ----- Load X, y depending on dataset -----
    if desired_dataset == "ecg":
        ECG_data_path = "../public_datasets/3D/ptb-xl-1.0.3"
        loader        = ECGLoader(ECG_data_path)
        X, y, _, _    = loader.load_dataset(sampling="lr", target="diagnostic_superclass_multi",
                                            segment_duration_sec=segment_duration_sec,
                                            max_records=max_records, continuous_target=True)
    elif desired_dataset == "argoverse":
        argoverse_data_path = "../public_datasets/3D/argoverse_forecasting"
        folder_name = "00a0ec58-1fb9-4a2b-bfd7-f4e5da7a9eff"
        file_name   = "scenario_00a0ec58-1fb9-4a2b-bfd7-f4e5da7a9eff.parquet"
        X, y        = process_argoverse_parquet(f"{argoverse_data_path}/{folder_name}/{file_name}")
    elif desired_dataset == "weather":
        dataset_folder = "../public_datasets/3D/weather_bench"
        variables_X    = ["2m_temperature", "10m_u_component_of_wind", "10m_v_component_of_wind"]
        variables_y    = ["mean_sea_level_pressure", "total_precipitation_6hr"]
        weather        = WeatherDataset(dataset_folder, variables_X, variables_y)
        X_xarr, y_xarr = weather.load_dataset()
        X_3d = weather.prepare_X_features(X_xarr, mode="3d")
        X_3d = weather.reshape_for_ml(X_3d)
        y_3d = weather.prepare_y_targets(y_xarr, mode="3d")
        y_2d = weather.flatten_y(y_3d)
        X, y = X_3d, y_2d
    elif desired_dataset == "india_catchment":
        data_path        = "../public_datasets/3D/india_catchments"
        forcing_folder   = "catchment_mean_forcings"
        clim_file        = "attributes_csv/camels_ind_clim.csv"
        y_df             = pd.read_csv(f"{data_path}/{clim_file}")
        catchment_ids    = y_df.iloc[:,0].astype(int).values
        y                = y_df.iloc[:, 1:]
        forcing_files    = sorted(os.listdir(f"{data_path}/{forcing_folder}"))
        X_list, file_ids = [], []
        for f in forcing_files:
            df = pd.read_csv(os.path.join(data_path, forcing_folder, f))
            X_list.append(df.drop(columns=['year','month','day','pet(mm/day)']).values)
            file_ids.append(int(f.split('.')[0]))
        X     = np.stack(X_list, axis=0)
        order = [file_ids.index(cid) for cid in catchment_ids]
        X     = X[order]
    elif desired_dataset == "germany_catchment":
        if os.path.exists(zarr_path):
            X = xr.open_zarr(zarr_path)["X"].values
        else:
            X = load_X_from_scratch(timeseries_folder, zarr_path)
        y = load_y_from_scratch(attributes_folder)
        if np.isnan(X).any():
            X = np.nan_to_num(X)
        if np.isnan(y).any():
            y = np.nan_to_num(y)
    elif desired_dataset == "china_weather":
        dataset_location = "../public_datasets/3D/china_weather" #"/Users/fouadabiad/Downloads/weather2k.npy"
        file_name        = "weather2k.npy"
        china_data       = np.load(os.path.join(dataset_location, file_name), mmap_mode='r')  # read-only memory map
        china_data       = china_data.transpose(0,2,1) # (stations, timesteps, features)
        y_indices = [4, 5, 6, 7, 10] # [T, mnt, mxt, rh, ws], to remove
        y         = china_data[:, -1, y_indices] # last timestep
        mask      = np.ones(china_data.shape[2], dtype=bool)
        mask[y_indices] = False
        X         = china_data[:, :, mask]
    else:
        raise ValueError(f"Unknown dataset: {desired_dataset}")

    # ----- Downsample + preprocess -----
    preprocessor = DatasetPreprocessor(page_frac=downsampling_dict[desired_dataset][0],
                                       row_frac=downsampling_dict[desired_dataset][1], test_size=0.2)
    result = preprocessor.fit_transform(X, y, split_X=split_X)
    return result

    # ----- Cast to float16 for memory -----
    # if split_X:
    #     X_train, X_test, y_train_scaled, y_test_scaled = result
    #     X_train = X_train.astype(np.float16)
    #     return X_train, X_test, y_train_scaled, y_test_scaled
    # else:
    #     X_small, y_train_scaled, y_test_scaled, _ = result
    #     X_small = X_small.astype(np.float16)
    #     return X_small, y_train_scaled, y_test_scaled, None

def load_or_preprocess(desired_dataset: str, segment_duration_sec: int = 150, max_records: int = 15000) -> tuple[np.ndarray, ...]:
    """Load cached preprocessed dataset if available, otherwise preprocess and cache it."""
    page_frac, row_frac = downsampling_dict[desired_dataset]
    new_dir_name        = f"{desired_dataset}_{page_frac}_{row_frac}"
    save_dir            = f"{interim_data_loc}/{new_dir_name}"
    
    if os.path.exists(save_dir): # Load from cache
        print("Path exists, loading from cache:", save_dir)
        X_train        = np.load(f"{save_dir}/X_train.npz")['data']
        X_test         = np.load(f"{save_dir}/X_test.npz")['data']
        y_train_scaled = np.load(f"{save_dir}/y_train.npz")['data']
        y_test_scaled  = np.load(f"{save_dir}/y_test.npz")['data']

        X_small        = np.load(f"{save_dir}/X_small.npz")['data']
        y_train_small  = np.load(f"{save_dir}/y_train_small.npz")['data']
        y_test_small   = np.load(f"{save_dir}/y_test_small.npz")['data']
    else: # Preprocess fresh
        print("Path does not exist, preprocessing fresh:", save_dir)
        os.makedirs(save_dir, exist_ok=True)

        X_train, X_test, y_train_scaled, y_test_scaled = load_and_preprocess_dataset(
            desired_dataset, split_X=True, segment_duration_sec=segment_duration_sec, 
            max_records=max_records)

        # Also keep a smaller version (for timevae)
        X_small, y_train_small, y_test_small, _ = load_and_preprocess_dataset(
            desired_dataset, split_X=False, segment_duration_sec=segment_duration_sec, 
            max_records=max_records)

        # Save to cache
        np.savez_compressed(f"{save_dir}/X_train.npz", data=X_train.astype(np.float32))
        np.savez_compressed(f"{save_dir}/X_test.npz", data=X_test.astype(np.float32))
        np.savez_compressed(f"{save_dir}/y_train.npz", data=y_train_scaled.astype(np.float32))
        np.savez_compressed(f"{save_dir}/y_test.npz", data=y_test_scaled.astype(np.float32))
        np.savez_compressed(f"{save_dir}/X_small.npz", data=X_small.astype(np.float32))
        np.savez_compressed(f"{save_dir}/y_train_small.npz", data=y_train_small.astype(np.float32))
        np.savez_compressed(f"{save_dir}/y_test_small.npz", data=y_test_small.astype(np.float32))
    return X_train, X_test, y_train_scaled, y_test_scaled, X_small, y_train_small, y_test_small

downsampling_dict = {"ecg": (0.05, 0.08),
                     "argoverse": (0.3, 0.1),
                     "weather": (0.3, 0.01),
                     "india_catchment": (0.3, 0.08),
                     "germany_catchment": (0.3, 0.015),
                     "china_weather": (0.25, 0.05)}

# desired_dataset = "china_weather" # options: ecg, argoverse, weather, india_catchment, germany_catchment, china_weather
baseline        = "timevae"  # options: "timevae", "ts2vec", "moment", "?", "?"

X_train, X_test, y_train_scaled, y_test_scaled,\
      X_small, y_train_small, y_test_small = load_or_preprocess(desired_dataset)

print(f"X_train: {X_train.shape} ({X_train.nbytes/1024**2:.2f} MB), X_test: {X_test.shape} ({X_test.nbytes/1024**2:.2f} MB)")
print(f"X_small: {X_small.shape} ({X_small.nbytes/1024**2:.2f} MB)")

page_frac, row_frac = downsampling_dict[desired_dataset]

if baseline == "timevae":
    os.makedirs(interim_data_loc, exist_ok=True)
    np.savez_compressed(f"../../timeVAE/data/{desired_dataset}_{page_frac}_{row_frac}.npz", data=np.array(X_small, dtype=np.float32))
    print(f"Saved {desired_dataset}_{page_frac}_{row_frac} to TimeVAE repo")


In [ ]:
"[RUN ME] functions for Headsup/cellsup"

def make_augmentations(X: torch.Tensor,  augment_type: str, device, scale_factor: float = 0.1) -> torch.Tensor:
    """Apply a chosen augmentation to a 3D time series batch.
    - X: Input array of shape (batch, time, channels).
    - augment_type: Type of augmentation to apply.
    - scale_factor: Magnitude factor controlling strength of augmentation.
    returns augmented array with the same shape as X (except cropping)"""
    batches, timesteps, cols = X.shape
    X = X.to(device)
    if augment_type == "jitter":
        noise = torch.randn_like(X) * float(scale_factor)
        return X + noise
    if augment_type == "scaling":
        # per-sample, per-channel scaling factor
        factor = torch.randn(batches, 1, cols, device=device) * scale_factor + 1.0
        return X * factor
    if augment_type == "mag_warp":
        mag = torch.empty(batches, 1, 1, device=device).uniform_(1 - scale_factor, 1 + scale_factor)
        return X * mag
    if augment_type == "time_warp":
        B, T, C = X.shape
        # random smooth warp along time
        tt = torch.linspace(-1, 1, T, device=device).unsqueeze(0).repeat(B, 1)  # (B,T)
        warp = tt + scale_factor * torch.randn(B, T, device=device)  # jittered time coords
        warp = warp.clamp(-1, 1)

        # make grid: need 2 coords (x,y), here y is dummy zero
        grid = torch.stack([warp, torch.zeros_like(warp)], dim=-1)  # (B,T,2)
        grid = grid.unsqueeze(2)  # (B,T,1,2)

        X_reshaped = X.permute(0, 2, 1).unsqueeze(-1)  # (B,C,T,1)
        X_warped   = F.grid_sample(X_reshaped, grid, mode="bilinear",
                                padding_mode="border", align_corners=True)
        return X_warped.squeeze(-1).permute(0, 2, 1)  # back to (B,T,C)
    if augment_type == "permutation":
        X_aug = torch.empty_like(X)
        # per-sample random segmentation + permutation
        for b in range(batches):
            n_segs   = int(torch.randint(2, 5, (1,)).item())
            pts      = np.linspace(0, timesteps, n_segs + 1, dtype=int)
            perm     = np.random.permutation(n_segs)
            pieces   = [X[b, pts[i]:pts[i+1], :] for i in perm]
            X_aug[b] = torch.cat(pieces, dim=0)
        return X_aug
    if augment_type == "cropping":
        keep    = int(timesteps * (1 - scale_factor))
        start   = int(torch.randint(0, timesteps - keep + 1, (1,)).item())
        cropped = X[:, start:start + keep, :]
        # pad or trim to keep shape (B, T, C)
        if cropped.shape[1] < timesteps:
            pad = torch.zeros(batches, timesteps - cropped.shape[1], cols, device=device)
            return torch.cat([cropped, pad], dim=1)
        else:
            return cropped
    if augment_type == "masking":
        mask  = (torch.rand(batches, timesteps, cols, device=device) < scale_factor)
        X_aug = X.clone()
        X_aug[mask] = 0.0
        return X_aug
    if augment_type == "drift":
        drift = torch.linspace(0, float(scale_factor), timesteps, device=device).view(1, timesteps, 1)
        sign  = 1.0 if torch.rand(1, device=device) < 0.5 else -1.0
        return X + sign * drift
    raise ValueError(f"Unknown augment type {augment_type}")

def _interpolate_to_length(x: torch.Tensor, target_len: int) -> torch.Tensor:
    """Interpolate along time dimension to match target length."""
    b, t, c = x.shape
    x = x.permute(0, 2, 1).unsqueeze(-1)  # (B, C, T, 1)
    x = F.interpolate(x, size=(target_len, 1), mode="linear", align_corners=True)
    return x.squeeze(-1).permute(0, 2, 1)  # (B, target_len, C)

def make_two_views_augmentation(X: torch.Tensor, device, scale: float = 0.1):
    aug_types = ["jitter", "scaling", "masking", "cropping", "time_warp"]
    a1, a2 = np.random.choice(aug_types, 2, replace=False)
    v1, v2 = make_augmentations(X, a1, device, scale), make_augmentations(X, a2, device, scale)
    # if cropping shortened, interpolate back
    if v1.shape[1] != X.shape[1]:
        v1 = _interpolate_to_length(v1, X.shape[1])
    if v2.shape[1] != X.shape[1]:
        v2 = _interpolate_to_length(v2, X.shape[1])
    return v1, v2

def compute_byol_loss(p_online: torch.Tensor, z_target: torch.Tensor) -> torch.Tensor:
    """Minimal BYOL loss: MSE between online predictions and target projections, adapted from the BYOL paper.
    Args:
        p_online: prediction from online network (batch, dim)
        z_target: projection from target network (batch, dim)
    Returns: Scalar loss"""
    # normalize for stability (optional but standard)
    p_online = F.normalize(p_online, dim=1)
    z_target = F.normalize(z_target, dim=1)

    z_target = z_target.detach() # stop gradients on target
    return 2 - 2 * (p_online * z_target).sum(dim=1).mean() # loss = MSE = 2 - 2 * cosine_sim

def update_target_encoding_ema(target_encoder: torch.nn.Module, online_encoder: torch.nn.Module, decay: float):
    """In-place EMA update of target params: target = decay*target + (1-decay)*online"""
    # return decay * target_encoding + (1 - decay) * new_values

    with torch.no_grad():
        for t_param, o_param in zip(target_encoder.parameters(), online_encoder.parameters()):
            t_param.data.mul_(decay).add_(o_param.data * (1.0 - decay))

def get_loss_weights(step, warmup_steps, max_steps): #can have schedule as linear OR cosine
    if step < warmup_steps:
        return dict(recon=1.0, contrast=0.0, pred=0.0)
    else:
        t = (step - warmup_steps) / (max_steps - warmup_steps)
        return dict(
            recon=max(0.1, 1.0 - t),   # decay recon to 0.1
            contrast=min(0.5, t),      # grow contrast to 0.5
            pred=min(1.0, t))           # grow pred to 1.0

def schedule_learning_rate(step, max_steps, lr_0=1e-3, lr_end=1e-5, schedule_type="linear"):
    """Compute learning rate at given step with linear or cosine decay from lr0 to lr_end."""
    if schedule_type == "linear":
        return lr_0 - (lr_0 - lr_end) * (step / max_steps)
    elif schedule_type == "cosine":
        cosine_decay = 0.5 * (1 + math.cos(math.pi * step / max_steps))
        return lr_end + (lr_0 - lr_end) * cosine_decay
    else:
        raise ValueError(f"Unknown schedule type: {schedule_type}")

def set_all_seeds(seed: int):
    import random

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def nt_xent_loss(z1, z2, temperature=0.5):
    """Normalized temperature-scaled cross entropy loss"""
    B = z1.size(0)   # dynamically set batch size
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    z = torch.cat([z1, z2], dim=0)  # (2B, dim)

    sim = torch.matmul(z, z.T) / temperature
    mask = torch.eye(2*B, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(mask, -9e15)

    labels = torch.cat([torch.arange(B) + B, torch.arange(B)], dim=0).to(z.device)
    loss = F.cross_entropy(sim, labels)
    return loss

def infer_catboost_multioutput(model: Optional[MultiOutputRegressor],
                               X_test: np.ndarray,
                               non_constant_idx: List[int],
                               n_targets: int,
                               fallback_values: Optional[np.ndarray] = None) -> np.ndarray:
    """Inference for multi-output CatBoost with support for constant targets."""
    y_pred = np.zeros((len(X_test), n_targets), dtype=float)

    if model is not None:
        preds = model.predict(X_test)
        y_pred[:, non_constant_idx] = preds

    if fallback_values is not None:
        for i in range(n_targets):
            if i not in non_constant_idx:  # constant column
                y_pred[:, i] = fallback_values[i]

    return y_pred

class LogRunResults:
    @staticmethod
    def log_run_in_csv(params: dict, results: dict, log_file: Path):
        """Logs a single experiment run.
        params: dictionary of all hyperparameters
        results: dictionary of evaluation metrics
        log_file: path to parquet or csv file"""
        row = {**params, **results, "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M")}
        
        if log_file.exists():
            df = pd.read_parquet(log_file) if log_file.suffix==".parquet" else pd.read_csv(log_file)
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        else:
            df = pd.DataFrame([row])
        
        if log_file.suffix==".parquet":
            df.to_parquet(log_file, index=False)
        else:
            df.to_csv(log_file, index=False)

    @staticmethod
    def log_run_json(params: dict, results_dict: dict, top_metrics: dict, log_file: Path):
        """Logs one experiment run as a JSON object with auto-incremented run_id.
        'results' contains both fraction results and top-level metrics, rounded to 4 decimals"""
        if log_file.exists():
            with open(log_file, "r") as f:
                run_id = sum(1 for _ in f) + 1
        else:
            run_id = 1

        # Merge fraction results and top-level metrics
        merged_results = {**{str(k): round(v, 4) for k, v in results_dict.items()},
                        **{k: round(v, 4) for k, v in top_metrics.items()}}

        run_data = {
            "run_id": run_id,
            **params,
            "results": merged_results,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M")}

        log_file.parent.mkdir(parents=True, exist_ok=True)
        with open(log_file, "a") as f:
            f.write(json.dumps(run_data, indent=4) + "\n")


class Preds():
    "Class of predictors to predict y from X"

    @staticmethod
    def predict_linreg(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray) -> float:
        """Train linear predictor"""
        model  = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        return root_mean_squared_error(y_test, y_pred)

    @staticmethod
    def predict_catboost_multioutput(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray) -> Tuple[Optional[MultiOutputRegressor], np.ndarray, float]:
        """Train multi-output CatBoost models and predict test set.
        Returns:
            model: trained MultiOutputRegressor (or None if all targets constant)
            y_pred: predictions on test set
            rmse: RMSE across all targets"""
        y_pred           = np.zeros_like(y_test, dtype=float)
        non_constant_idx = [i for i in range(y_train.shape[1])
                            if not np.all(y_train[:, i] == y_train[0, i])]
        if non_constant_idx:
            model = MultiOutputRegressor(CatBoostRegressor(iterations=500, learning_rate=0.1, depth=4, verbose=0))
            model.fit(X_train, y_train[:, non_constant_idx])
            y_pred[:, non_constant_idx] = model.predict(X_test)
            for i in range(y_train.shape[1]):
                if np.all(y_train[:, i] == y_train[0, i]):
                    y_pred[:, i] = y_train[0, i]
        else:
            model = None
        rmse = root_mean_squared_error(y_test, y_pred)
        return model, y_pred, rmse, non_constant_idx

    @staticmethod
    def cluster_and_label(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray,
                        y_test: np.ndarray, n_clusters: int = 10, random_state: int = 42) -> float:
        """Cluster train features with KMeans, assign representative y (mean per cluster), predict test labels by cluster assignment,
        and compute RMSE. this is considered unsupervised, as the clustering happens to X only
        - X_train: Training features (N_train, D).
        - y_train: Training targets (N_train, T).
        - X_test: Test features (N_test, D).
        - y_test: Test targets (N_test, T).
        - n_clusters: Number of KMeans clusters.
        - random_state: Random seed for reproducibility.
        Returns: Mean squared error on test set."""
        try:
            kmeans         = KMeans(n_clusters=n_clusters, random_state=random_state)
            train_clusters = kmeans.fit_predict(X_train)

            # mean target vector per cluster
            y_cluster     = {cluster: y_train[train_clusters == cluster].mean(axis=0)
                            for cluster in range(n_clusters)}
            test_clusters = kmeans.predict(X_test)
            y_pred        = np.stack([y_cluster[cluster] for cluster in test_clusters], axis=0)
            return root_mean_squared_error(y_test, y_pred)
        except Exception:
            return float('nan')

    @staticmethod
    def predict_rf_multioutput(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray) -> float:
        """Train multi-output Random Forest and compute RMSE."""
        model  = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        return root_mean_squared_error(y_test, y_pred)

    @staticmethod
    def predict_elasticnet_multioutput(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray,
                                       alpha: float = 0.1, l1_ratio: float = 0.5) -> Tuple[MultiOutputRegressor, np.ndarray, float]:
        """Train multi-output ElasticNet (L1+L2) and predict test set.
        Returns:
            model: trained MultiOutputRegressor
            y_pred: predictions on test set
            rmse: root mean squared error"""
        model  = MultiOutputRegressor(ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=1000, random_state=42))
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        rmse   = root_mean_squared_error(y_test, y_pred)
        return model, y_pred, rmse

    @staticmethod
    def predict_mlp_multioutput(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray,
                                hidden_layer_sizes: Tuple[int, ...] = (128, 64), max_iter: int = 500,
                                random_state: int = 42) -> Tuple[MultiOutputRegressor, np.ndarray, float]:
        """Train multi-output MLPRegressor and predict test set.
        Returns:
            model: trained MultiOutputRegressor
            y_pred: predictions on test set
            rmse: root mean squared error"""
        base_mlp = MLPRegressor(hidden_layer_sizes=hidden_layer_sizes,
                                max_iter=max_iter,
                                random_state=random_state)
        model    = MultiOutputRegressor(base_mlp)
        model.fit(X_train, y_train)
        y_pred   = model.predict(X_test)
        rmse     = root_mean_squared_error(y_test, y_pred)
        return model, y_pred, rmse

    @staticmethod
    def evaluate_models_on_dataset(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray):
        """Evaluate various models on the dataset and print RMSE results."""
        linreg_loss       = Preds.predict_linreg(X_train, y_train, X_test, y_test)
        _, _, catboost_loss, _ = Preds.predict_catboost_multioutput(X_train, y_train, X_test, y_test)
        # unsupervised_rmse = Preds.cluster_and_label(X_train, y_train, X_test, y_test, n_clusters=5)
        rf_rmse           = Preds.predict_rf_multioutput(X_train, y_train, X_test, y_test)
        # _, _, el_rmse     = Preds.predict_elasticnet_multioutput(X_train, y_train, X_test, y_test, alpha=0.1, l1_ratio=0.5)
        return linreg_loss, catboost_loss, rf_rmse,# unsupervised_rmse #, el_rmse



In [ ]:
"""TimeVAE"""

if params["run_console"]["timevae"] == True:
    z_train     = np.load(f"{interim_data_loc}/z_train_{desired_dataset}.npy")
    z_valid     = np.load(f"{interim_data_loc}/z_valid_{desired_dataset}.npy")
    z_test      = np.load(f"{interim_data_loc}/z_test_{desired_dataset}.npy")
    z_train_val = np.concatenate([z_train, z_valid], axis=0) #concat z_train and val to = y shape
    print(f"z_train: {z_train.shape}, z_valid: {z_valid.shape}, z_train_val: {z_train_val.shape}, z_test: {z_test.shape}")
    print(f"y_train_scaled: {y_train_scaled.shape}, y_test_scaled: {y_test_scaled.shape}")

    # Train NN predictor
    predictor_lr      = params["timevae"]["predictor_lr"]
    predictor_epochs  = params["timevae"]["predictor_epochs"]
    predictor_dropout = params["timevae"]["predictor_dropout"]
    predictor_hidden_sizes = params["timevae"]["predictor_hidden_sizes"]

    nn_predictor = MLPHead(input_dim=z_train.shape[1], output_dim=y_train_scaled.shape[1],
                           hidden_sizes=predictor_hidden_sizes, lr=predictor_lr,
                           epochs=predictor_epochs, dropout=predictor_dropout, device=device)
    nn_predictor.train(z_train_val, y_train_scaled, z_test, y_test_scaled)
    nn_rmse = nn_predictor.evaluate(z_test, y_test_scaled)

    # ====== plain predictors ======
    timevae_losses = Preds.evaluate_models_on_dataset(z_train_val, y_train_scaled,
                                                      z_test, y_test_scaled)
    print(f"dataset: {desired_dataset}, method: timevae")
    print( "    RMSE      | LinReg | CatBoost | RForest | NN")
    print(f"& Z (timevae) & {timevae_losses[0]:.4f} & {timevae_losses[1]:.4f}  & {timevae_losses[1]:.4f} \
        & {nn_rmse:.4f} \\ ")


In [ ]:
"""TS2Vec"""
if params["run_console"]["ts2vec"] == True:

    z_pooling_method   = params["ts2vec"]["z_pooling_method"]
    ts2vec_hidden_dims = params["ts2vec"]["ts2vec_hidden_dims"]
    ts2vec_latent_dims = params["ts2vec"]["ts2vec_latent_dims"]
    ts2vec_depth       = params["ts2vec"]["ts2vec_depth"]
    ts2vec_batch_size  = params["ts2vec"]["ts2vec_batch_size"]
    ts2vec_epochs      = params["ts2vec"]["ts2vec_epochs"]

    predictor_lr       = params["ts2vec"]["predictor_lr"]
    predictor_epochs   = params["ts2vec"]["predictor_epochs"]
    predictor_dropout  = params["ts2vec"]["predictor_dropout"]
    predictor_hidden_sizes = params["ts2vec"]["predictor_hidden_sizes"]

    # 1️⃣ Fit TS2Vec
    ts2vec_encoder = TS2VecEncoder(z_pooling=z_pooling_method, device=device, patience=20)  # keep patience
    ts2vec_encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
                    depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)
    if ts2vec_encoder._stop_early:
        print("Training stopped early due to no improvement.")
    z_train = ts2vec_encoder.encode(X_train)
    z_test  = ts2vec_encoder.encode(X_test)

    # 2️⃣ Train NN predictor
    nn_predictor = MLPHead(input_dim=z_train.shape[1], output_dim=y_train_scaled.shape[1],
                        hidden_sizes=predictor_hidden_sizes, lr=predictor_lr,
                        epochs=predictor_epochs, dropout=predictor_dropout, device=device)
    nn_predictor.train(z_train, y_train_scaled, z_test, y_test_scaled)
    nn_rmse = nn_predictor.evaluate(z_test, y_test_scaled)

    # ====== plain predictors ======
    ts2vec_losses = Preds.evaluate_models_on_dataset(z_train, y_train_scaled, z_test, y_test_scaled)
    print(f"dataset: {desired_dataset}, method: ts2vec")
    print("    RMSE   | LinReg | CatBoost | RForest | NN")
    print(f"& Z (ts2vec) & {ts2vec_losses[0]:.4f} & {ts2vec_losses[1]:.4f} & {ts2vec_losses[2]:.4f} & {nn_rmse:.4f} \\ ")


In [ ]:
"""MOMENT"""

if params["run_console"]["moment"] == True:
    from momentfm import MOMENTPipeline

    model_type        = params["moment"]["model_type"] # options: classification (= regression), repres_learning
    model_name        = params["moment"]["model_name"] #"MOMENT-1-large"
    reload_model      = params["moment"]["reload_model"]
    predictor_lr      = params["moment"]["predictor_lr"]
    predictor_epochs  = params["moment"]["predictor_epochs"]
    predictor_dropout = params["moment"]["predictor_dropout"]
    predictor_hidden_sizes = params["moment"]["predictor_hidden_sizes"]

    if reload_model == True:
        if model_type == "classification": # classification/regression
            classes_in_y = len(np.unique(y_train_scaled))
            moment_model = MOMENTPipeline.from_pretrained(f"AutonLab/{model_name}", 
                                                        model_kwargs={'task_name': 'classification',
                                                                        'n_channels': X_train.shape[2], 'num_class': classes_in_y},)
        elif model_type == "repres_learning": # representation learning
            moment_model = MOMENTPipeline.from_pretrained(f"AutonLab/{model_name}", 
                                                        model_kwargs={"task_name": "embedding"},)
        print(f"Doing {model_type} with {model_name}!")
        moment_model.init()
        moment_model.eval()

    # 1️⃣ Encode train and test separately (NOTE: model expects data as [batch, channels, seq_len] )
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).permute(0, 2, 1)
    X_test_tensor  = torch.tensor(X_test, dtype=torch.float32).permute(0, 2, 1)

    with torch.no_grad():
        train_outputs = moment_model(x_enc=X_train_tensor)
        test_outputs  = moment_model(x_enc=X_test_tensor)

    z_train = train_outputs.embeddings.cpu().numpy()
    z_test  = test_outputs.embeddings.cpu().numpy()

    if model_type == "classification":
        z_train = z_train.mean(axis=1)  # or max(axis=1)
        z_test  = z_test.mean(axis=1)

    print(X_train.shape, z_train.shape)

    # 2️⃣ Train downstream NN predictor
    nn_predictor = MLPHead(input_dim=z_train.shape[1], output_dim=y_train_scaled.shape[1],
                            hidden_sizes=predictor_hidden_sizes, lr=predictor_lr,
                            epochs=predictor_epochs, dropout=predictor_dropout, device=device)
    nn_predictor.train(z_train, y_train_scaled, z_test, y_test_scaled)
    nn_rmse = nn_predictor.evaluate(z_test, y_test_scaled)

    # ====== plain predictors ======
    moment_losses = Preds.evaluate_models_on_dataset(z_train, y_train_scaled, z_test, y_test_scaled)
    print(f"dataset: {desired_dataset}, method: moment")
    print( "    RMSE   | LinReg | CatBoost | RForest | NN")
    print(f"& Z ({model_type}) & {moment_losses[0]:.4f} & {moment_losses[1]:.4f} & {moment_losses[2]:.4f} \
        & {nn_rmse:.4f} \\ ")


In [ ]:
"params for running Headsup"
# ===== TS2Vec params ======
z_pooling_method   = "mean"
ts2vec_hidden_dims = 16 # units in each layer (> than latent dim)
ts2vec_latent_dims = 8 # latent dim
ts2vec_depth       = 3 # num layers
ts2vec_batch_size  = 32
ts2vec_epochs      = 5 #20
ts2vec_patience    = 25

# predictor_lr           = 0.009
# predictor_epochs       = 50
predictor_dropout      = 0.05
predictor_hidden_sizes = [32, 48, 64] # latent to y output

# ===== Headsup pretrain/train params =====
set_all_seeds(42)
batch_size_pretrain  = 16
batch_size_train     = 16

decoder_hidden_dims  = [16, 64, 128]
projection_dim       = 16
lr_pretrain          = 1e-3
lr_train             = 1e-4 #1e-3

patience_pretrain    = 15
patience_train       = 6

train_epochs_pretrain= 8
train_epochs_finetune= 3 # aim for 30-50
warmup_frac_pretrain = 0.05 # 5-10% of pretrain steps
warmup_frac_train    = 0.05 # 5-10% of train steps

weights_pretrain     = {"recon":0.1,"contrast":1.0}
weights_train_100    = {"pred": 1.0, "recon": 0.5, "contrast": 0.5} # 100 is the label_fraction
weights_train_50     = {"pred": 1.0, "recon": 0.1, "contrast": 0.1} # 50 is the label_fraction
weights_train_other  = {"pred": 2.0, "recon": 0.0, "contrast": 0.0}

label_fractions = [1.0, 0.5, 0.25, 0.1]

# === augmentations ===
aug1          = "jitter"
aug1_strength = 0.2
aug2          = "mag_warp"
aug2_strength = 0.1

# === files and names ===
TS2VEC_ENCODER_NAME   = f"ts2vec_encoder_{desired_dataset}_{ts2vec_hidden_dims}hiddendims_{ts2vec_depth}layers_{ts2vec_latent_dims}dims_{ts2vec_batch_size}batch_{ts2vec_epochs}epoch.pkl"
TS2VEC_ENCODER_FILE   = os.path.join(interim_data_loc, "ts2vec_encoders", TS2VEC_ENCODER_NAME)

PRETRAIN_ENCODER_NAME = (f'pretrained_encoder_{desired_dataset}_lr{lr_pretrain}_epochs{train_epochs_pretrain}_batch{batch_size_pretrain}'
                         f'_enc{ts2vec_latent_dims}dims_{ts2vec_depth}layers_dec{decoder_hidden_dims}_proj{projection_dim}'
                         f'_warmup{warmup_frac_pretrain}_frac{"_".join(map(str, label_fractions))}.pth')
PRETRAIN_ENCODER_FILE = os.path.join(interim_data_loc, "pretrained_encoders", PRETRAIN_ENCODER_NAME)

EMBEDDING_FILE_NAME   = (f"cached_embeddings_{desired_dataset}_{desired_dataset}_lr{lr_train}_epochs{train_epochs_finetune}_batch{batch_size_train}"
                         f'_enc{ts2vec_latent_dims}dims_{ts2vec_depth}layers_dec{decoder_hidden_dims}_proj{projection_dim}'
                         f'_warmup{warmup_frac_train}.pth')
EMBEDDING_CACHE_FILE  = os.path.join(interim_data_loc, "trained_encoders", EMBEDDING_FILE_NAME)


In [ ]:
"""Cellsup functions"""
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

def get_latent_from_encoder(encoder, X, device="cpu") -> np.ndarray:
    """Return latent z for any encoder type with shape (N, latent_dim)."""
    if isinstance(encoder, nn.Module):
        X_tensor = torch.tensor(X, dtype=torch.float32, device=device)
        if X_tensor.ndim > 2:
            X_tensor = X_tensor.reshape(len(X_tensor), -1)
        if type(encoder).__name__ == "VAE":
            mu, _ = encoder.encode(X_tensor)
            z     = mu.detach().cpu().numpy()
        else: # AE / DAE: return latent layer instead of reconstruction
            z = encoder.encode(X_tensor).detach().cpu().numpy()
    elif type(encoder).__name__ == "TS2VecEncoder":
        z = encoder.encode(X)  # returns (N, T, latent_dim)
        z = z.mean(axis=1)      # temporal pooling
    else:
        raise ValueError(f"Unknown encoder type: {type(encoder).__name__}")
    return z

def bootstrap_sample(X, sample_frac=0.8):
    """Draw a bootstrap sample from X with replacement. Used to approximate sampling variability when the
    true population dist is unknown
        - X: Input array of shape (n_samples, ...).
        - sample_frac: Fraction of samples to draw (default=0.8).
        - returns: bootstrap sample array of shape (int(n_samples * sample_frac), ...)"""
    idx = np.random.choice(len(X), size=int(len(X)*sample_frac), replace=True)
    return X[idx]

def train_ae_with_bootstraps(model, X_train, num_epochs=5, lr=1e-3, sample_frac=0.8, weight_decay=0.0, device="cpu"):
    """Train AE with bootstrap sampling."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    X_tensor_full = torch.tensor(X_train, dtype=torch.float32, device=device).reshape(len(X_train), -1)
    
    for epoch in range(num_epochs):
        X_boot = bootstrap_sample(X_train, sample_frac)
        X_tensor_boot = torch.tensor(X_boot, dtype=torch.float32, device=device).reshape(len(X_boot), -1)
        optimizer.zero_grad()
        X_recon = model(X_tensor_boot)
        loss = F.mse_loss(X_recon, X_tensor_boot)
        loss.backward()
        optimizer.step()
    return model

def flatten_X(X: np.ndarray) -> np.ndarray:
    """Flatten 2D or 3D X to (N, D) for torch feeding."""
    if X.ndim > 2:
        return X.reshape(len(X), -1)
    return X

def get_latent_tensor(encoder, X, train_encoder=False, device="cpu") -> torch.Tensor:
    """Return 2D tensor (N, latent_dim) for SupHead/CatBoost."""
    if train_encoder and isinstance(encoder, nn.Module):
        encoder.train()
        X_tensor = torch.tensor(flatten_X(X), dtype=torch.float32, device=device)
        z = encoder.encode(X_tensor)
        if isinstance(z, tuple):  # for VAE
            z = z[0]
        if z.ndim > 2:
            z = z.mean(dim=1)
        return z
    else:
        z = get_latent_from_encoder(encoder, X, device=device)
        if z.ndim > 2:
            z = z.mean(axis=1)
        return torch.tensor(z, dtype=torch.float32, device=device) if isinstance(z, np.ndarray) else z

def _get_orthogonality_penalty(encoders, X_batch, device):
    """Compute sum of squared correlations between encoder latent batches.
       encoders: dict[name]->encoder; X_batch: same X for all encoders (or views applied externally)"""
    z_list = []
    for enc in encoders.values():
        z = get_latent_tensor(enc, X_batch, train_encoder=False, device=device)  # (B, d)
        z = z - z.mean(0)
        # l2-normalize per feature to reduce scale issues
        z = z / (z.std(0) + 1e-8)
        z_list.append(z)  # torch tensors
    # compute pairwise dot products of mean latent vectors (or flattened)
    penalty = 0.0
    for i in range(len(z_list)):
        for j in range(i+1, len(z_list)):
            # compute covariance between latent dims (sum of squared correlations)
            C = (z_list[i].T @ z_list[j]) / z_list[i].shape[0]  # (d_i, d_j)
            penalty = penalty + (C ** 2).sum()
    return penalty

def train_sup_head_per_encoder(encoder, X_L, y_L, X_test, y_test, dropout, train_encoder=True,
                               all_encoders=None, reg_ortho=5e-4,   # tune this
                               device="cpu", epochs=5, hidden_sizes=[64,32]):
    """Train a small supervised MLP head on top of EACH encoder's z.
    Args:
        encoder: AE/VAE/DAE/TS2Vec encoder
        X_L, y_L: labelled training data
        X_test, y_test: test data
        train_encoder: whether to finetune encoder
        device: "cpu"/"cuda"
        epochs: training epochs
        hidden_sizes: list of hidden layer sizes
    Returns:
        rmse on test set"""
    encoder_type  = type(encoder).__name__
    train_encoder = train_encoder and isinstance(encoder, nn.Module) and encoder_type in ["FlexibleAutoencoder","AE","VAE","DenoisingAE"]

    encoder.eval()
    z_sample   = get_latent_tensor(encoder, X_L[:2], train_encoder=False, device=device)
    latent_dim = z_sample.shape[-1]
    sup_head   = SupHead(latent_dim, y_L.shape[1], hidden_sizes, dropout).to(device)
    params     = list(sup_head.parameters())
    if train_encoder:
        params += list(encoder.parameters())
        # =========
        for other_name, other_enc in all_encoders.items():
            if other_enc is not encoder:
                params += list(other_enc.parameters())
        # =========
    optimizer  = torch.optim.AdamW(params, lr=1e-3)
    y_tensor   = torch.tensor(y_L, dtype=torch.float32, device=device)
    
    for _ in range(epochs):
        sup_head.train()
        if train_encoder:
            encoder.train()
        z      = get_latent_tensor(encoder, X_L, train_encoder=train_encoder, device=device)
        y_pred = sup_head(z)
        loss   = F.mse_loss(y_pred, y_tensor)

        # ====== add orthogonality penalty across encoder ensemble (very cheap)
        if all_encoders is not None and reg_ortho > 0:
            # use a small random batch for speed
            idx       = np.random.choice(len(X_L), size=min(128, len(X_L)), replace=False)
            X_batch   = X_L[idx]
            ortho_pen = _get_orthogonality_penalty(all_encoders, X_batch, device=device)
            loss      = loss + reg_ortho * ortho_pen
        # ======

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    sup_head.eval()
    with torch.no_grad():
        z_test      = get_latent_tensor(encoder, X_test, train_encoder=False, device=device)
        y_pred_test = sup_head(z_test).cpu().numpy()
    return root_mean_squared_error(y_test, y_pred_test)

def train_sup_heads_joint(encoders_dict, X_train, y_train, X_val, y_val,
                          hidden_sizes=[64], lr=0.001, epochs=50, device="cpu",
                          reg_ortho=1e-3, train_encoders=True, batch_size=128):
    """Jointly train supervised heads for each encoder with optional finetuning and orthogonality.
    Updates encoders_dict in-place with finetuned encoders.
    Returns dict of RMSE metrics."""
    metrics = {}
    for name, encoder in encoders_dict.items():
        rmse = train_sup_head_per_encoder(
            encoder, X_train, y_train, X_val, y_val,
            dropout=0.1, train_encoder=train_encoders,
            all_encoders=encoders_dict, reg_ortho=reg_ortho,
            device=device, epochs=epochs, hidden_sizes=hidden_sizes)
        metrics[name] = rmse
    return metrics

def train_and_eval_catboost(X_train, y_train, X_test, y_test):
    """Train MultiOutput CatBoost, handle constant columns, return predictions and RMSE."""
    mask       = [i for i in range(y_train.shape[1]) if not np.all(y_train[:, i] == y_train[0,i])]
    const_vals = {i: y_train[0,i] for i in range(y_train.shape[1]) if i not in mask}

    y_pred = np.zeros_like(y_test)
    if mask:
        model = MultiOutputRegressor(CatBoostRegressor(iterations=500, learning_rate=0.1, depth=4,
                                                       random_seed=42, verbose=0))
        model.fit(X_train[:, :], y_train[:, mask])
        y_pred[:, mask] = model.predict(X_test)

    for i, v in const_vals.items():
        y_pred[:, i] = v

    rmse = root_mean_squared_error(y_test, y_pred)
    return y_pred, rmse

def assign_encoder_weights(encoders_dict: dict, sup_head_rmse, weight_encoding_method: str = "uniform"):
    """Compute normalized encoder weights using one of three methods:
        - "uniform": equal weights
        - "inverse_rmse": proportional to 1/RMSE
        - "softmax": softmax over 1/RMSE"""
    if weight_encoding_method == "uniform":
        encoder_weights = {name: 1.0 for name in encoders_dict.keys()}
        total           = sum(encoder_weights.values())
        encoder_weights = {k: v / total for k, v in encoder_weights.items()}
    elif weight_encoding_method == "inverse_rmse": # RMSE-based weights: better encoders get higher weight
        encoder_weights = {name: 1/rmse for name, rmse in sup_head_rmse.items()}
        total           = sum(encoder_weights.values())
        encoder_weights = {k: v/total for k,v in encoder_weights.items()}
    elif weight_encoding_method == "softmax": # softmax-based weights
        inv_rmse        = np.array([1/r for r in sup_head_rmse.values()])
        weights_softmax = np.exp(inv_rmse) / np.sum(np.exp(inv_rmse))
        encoder_weights = {name: w for name, w in zip(sup_head_rmse.keys(), weights_softmax)}
    return encoder_weights

def target_distribution(q: np.ndarray) -> np.ndarray:
    """Sharpen soft assignments q -> p (DEC target distribution)."""
    weight = (q ** 2) / q.sum(axis=0)
    return (weight.T / weight.sum(axis=1)).T


class SupHead(nn.Module):
    """Small supervised head: maps latent z -> target y"""
    def __init__(self, input_dim: int, output_dim: int, hidden_sizes=[64, 32], dropout=1e-3):
        super().__init__()
        layers, prev_dim = [], input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)

class Cellsup:
    """Ensemble clustering across multiple encoders."""
    def __init__(self, encoders_dict: dict, n_clusters: int = 5, device: str = "cpu",
                 cluster_assignment: str = "soft", cluster_metric: str = "ch", random_state: int = 42):
        self.encoders_dict      = encoders_dict
        self.n_clusters         = n_clusters
        self.device             = device
        self.cluster_assignment = cluster_assignment
        self.cluster_prob_matrix= None
        self.cluster_metric     = cluster_metric.lower()
        self.rng                = np.random.RandomState(random_state)
        self.clusterers         = {}  # stores KMeans per encoder

    def score_clusters(self, z: np.ndarray, labels: np.ndarray) -> float:
        """Return clustering quality score depending on metric."""
        try:
            if self.cluster_metric == "silhouette":
                return silhouette_score(z, labels)
            elif self.cluster_metric == "ch":
                return calinski_harabasz_score(z, labels)
            elif self.cluster_metric == "db":
                return -davies_bouldin_score(z, labels)  # negate so higher = better
            else:
                raise ValueError(f"Unknown metric {self.cluster_metric}")
        except ValueError:
            return -1  # invalid clustering (e.g. only 1 cluster found)

    def _fit_best_kmeans_for_latent(self, z: np.ndarray, cluster_range: tuple[int,int], n_restarts: int = 5) -> KMeans:
        """Fit KMeans on latent `z` using separate best-k selection per encoder.
        1) Tune best cluster number k (average score over n_restarts)
        2) Fit KMeans with n_restarts for that k to select the best initialization
        Args:
            z (np.ndarray): Latent representations, shape (N, D)
            cluster_range (tuple[int,int]): Min and max+1 clusters to try
            n_restarts (int): KMeans restarts per candidate cluster number and for final best k
        Returns:
            KMeans: Fitted KMeans model with optimal cluster number"""
        # Step 1: pick best k using mean score across n_restarts
        best_score, best_k = -np.inf, None
        for k in range(cluster_range[0], cluster_range[1]):
            scores = []
            for _ in range(n_restarts):
                kmeans_try = KMeans(n_clusters=k, random_state=self.rng.randint(0,10000)).fit(z)
                scores.append(self.score_clusters(z, kmeans_try.labels_))
            mean_score = np.mean(scores)
            if mean_score > best_score:
                best_score, best_k = mean_score, k

        # Step 2: fit KMeans with multiple restarts for that best k
        best_score_restart, best_kmeans = -np.inf, None
        for _ in range(n_restarts):
            kmeans = KMeans(n_clusters=best_k, random_state=self.rng.randint(0,10000)).fit(z)
            score  = self.score_clusters(z, kmeans.labels_)
            if score > best_score_restart:
                best_score_restart, best_kmeans = score, kmeans

        print(f"Selected cluster size: {best_kmeans.n_clusters}")
        return best_kmeans

    def fit_kmeans_on_encoder_latents(self, X: np.ndarray, encoder_weights: dict = None,
                                      cluster_range=(4,16)):
        """Fit KMeans on each encoder's latent space, select the best cluster number per encoder,
        and concatenate per-encoder cluster features into a single matrix.
        Cluster features:
        - 'soft': standardized softmax of negative distances (approx. probabilities)
        - 'hard': one-hot cluster labels
        Optional encoder weights scale each encoder's features.
        Args:
            X (np.ndarray): Input data, shape (N, D) or (N, T, D)
            encoder_weights (dict, optional): Encoder name → scalar weight (normalized)
            cluster_range (tuple, optional): Range of clusters to try (min, max+1)
        Returns:
            self: Adds `self.clusterers` (dict of fitted KMeans) and 
                `self.prob_matrix` (concatenated cluster features, shape (N, sum_k))"""
        cluster_features_list = []

        if encoder_weights is not None: # normalize encoder weights if provided
            total = sum(encoder_weights.values())
            encoder_weights = {k: v / total for k, v in encoder_weights.items()}

        for name, encoder in self.encoders_dict.items():
            z = get_latent_from_encoder(encoder, X, device=self.device)
            kmeans               = self._fit_best_kmeans_for_latent(z, cluster_range, n_restarts=5)
            self.clusterers[name]= kmeans
            labels               = kmeans.labels_

            # --- compute soft/hard assignments ---
            if self.cluster_assignment == "soft":
                distances        = kmeans.transform(z)
                cluster_features = softmax(-distances, axis=1)
                # standardize to prevent dominance by encoders with more clusters
                cluster_features = (cluster_features - cluster_features.mean(axis=0)) / (cluster_features.std(axis=0) + 1e-8)
            else: # hard
                cluster_features = np.eye(kmeans.n_clusters)[labels]

            if encoder_weights is not None: # apply encoder weight
                cluster_features *= encoder_weights.get(name, 1.0)
            cluster_features_list.append(cluster_features)

        # concatenate horizontally across encoders
        self.cluster_prob_matrix = np.concatenate(cluster_features_list, axis=1)  # (N, sum_k)
        return self

    def apply_kmeans_to_new_data(self, X: np.ndarray) -> np.ndarray:
        """Apply EXISTING fitted KMeans models to new data X and return concat cluster features
        Cluster features:
        - 'soft': softmax of negative distances
        - 'hard': one-hot labels
        Args: X (np.ndarray): Input data, shape (N, D) or (N, T, D)
        Returns: np.ndarray: Concatenated cluster features, shape (N, sum_k)"""
        cluster_features_list = []
        for name, encoder in self.encoders_dict.items():
            z      = get_latent_from_encoder(encoder, X, device=self.device)
            kmeans = self.clusterers[name]
            labels = kmeans.predict(z)
            if self.cluster_assignment == "soft":
                distances        = kmeans.transform(z) # (n_samples, n_clusters)
                cluster_features = softmax(-distances, axis=1) # convert distance → probability
            elif self.cluster_assignment == "hard":
                cluster_features = np.eye(kmeans.n_clusters)[labels]
            cluster_features_list.append(cluster_features)
        # return np.mean(cluster_features_list, axis=0)
        # fix
        return np.concatenate(cluster_features_list, axis=1)  # (N, E*K)

    def assign_pseudo_labels(self, X_labeled, y_labeled, X_unlabeled, confidence_thresh=0.5):
        """Assign pseudo-labels to unlabeled data using KMeans cluster averages."""
        pseudo_labels_all = []

        if X_unlabeled is None or len(X_unlabeled) == 0:
            return np.zeros((0, y_labeled.shape[1]))

        for name, kmeans in self.clusterers.items():
            z_L      = get_latent_from_encoder(self.encoders_dict[name], X_labeled, device=device)
            z_U      = get_latent_from_encoder(self.encoders_dict[name], X_unlabeled, device=device)
            labels_L = kmeans.predict(z_L)
            labels_U = kmeans.predict(z_U)

            # cluster -> mean label map
            cluster_to_mean = {c: y_labeled[labels_L == c].mean(axis=0) for c in np.unique(labels_L)}
            fallback = y_labeled.mean(axis=0)  # use global mean instead of NaNs
            y_pseudo = np.stack([cluster_to_mean.get(c, fallback) for c in labels_U], axis=0)
            # ==============
            # # compute soft assignment probabilities
            # distances  = kmeans.transform(z_U)
            # soft_probs = softmax(-distances, axis=1)
            # max_probs  = soft_probs.max(axis=1)
            # # mask low-confidence pseudo-labels
            # mask = max_probs >= confidence_thresh
            # y_pseudo[~mask] = np.nan  # mark low-confidence samples as NaN
            # ==============
            pseudo_labels_all.append(y_pseudo)

        # combine across encoders
        y_pseudo_final = np.nanmean(np.stack(pseudo_labels_all, axis=0), axis=0)
        return y_pseudo_final

    # to replace the above one
    def assign_ensemble_pseudo_labels(self, X_labeled, y_labeled, X_unlabeled,
                                      cluster_sizes=(4, 6, 8), encoder_weights=None):
        """Assign pseudo-labels by averaging across multiple cluster sizes and encoders.
        Args:
            X_labeled (np.ndarray): labeled features
            y_labeled (np.ndarray): labels
            X_unlabeled (np.ndarray): unlabeled features
            cluster_sizes (tuple[int]): cluster numbers to ensemble
            encoder_weights (dict, optional): encoder name -> scalar weight
        Returns:
            np.ndarray: pseudo-labels for unlabeled data (shape: n_samples, y_dim)"""
        all_pseudo_labels = []

        for name, encoder in self.encoders_dict.items():
            z_L = get_latent_from_encoder(encoder, X_labeled, device=self.device)
            z_U = get_latent_from_encoder(encoder, X_unlabeled, device=self.device)

            for k in cluster_sizes:
                kmeans = KMeans(n_clusters=k, random_state=42).fit(z_L)
                labels_L = kmeans.predict(z_L)
                labels_U = kmeans.predict(z_U)

                cluster_to_mean = {c: y_labeled[labels_L == c].mean(axis=0) for c in np.unique(labels_L)}
                fallback = y_labeled.mean(axis=0)
                y_pseudo = np.stack([cluster_to_mean.get(c, fallback) for c in labels_U], axis=0)

                # optional encoder weight
                if encoder_weights is not None:
                    y_pseudo *= encoder_weights.get(name, 1.0)
                all_pseudo_labels.append(y_pseudo)

        # ensemble average across encoders and cluster sizes
        return np.nanmean(np.stack(all_pseudo_labels, axis=0), axis=0)

    # consider removing
    def refine_clusters_DEC(self, X, encoder, name: str, n_iters: int = 10, lr: float = 1e-4):
        """Refine encoder so that latent z matches clusters better (DEC refinement).
        Assumes encoder is a torch.nn.Module."""
        optimizer = torch.optim.Adam(encoder.parameters(), lr=lr)

        # get cluster centers from fitted KMeans
        # centers = torch.tensor(self.clusterers['AE_8'].cluster_centers_, dtype=torch.float32)
        centers = torch.tensor(self.clusterers[name].cluster_centers_, dtype=torch.float32)

        for _ in range(n_iters):
            z = get_latent_tensor(encoder, X, train_encoder=True, device=self.device)  # (n, d)

            # soft assignment of z to centers
            q = torch.softmax(-torch.cdist(z, centers.to(z.device)), dim=1)

            # sharpened target distribution
            p = torch.tensor(target_distribution(q.detach().cpu().numpy()), device=z.device)

            # KL divergence loss
            loss = F.kl_div(q.log(), p, reduction="batchmean")

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # to replace with SWAV
    def deepcluster_step(self, X, n_iters=1, refine_encoder=False, lr=1e-3, epochs_finetune=5):
        """Iterative DeepCluster-style for lightweight modular pseudo-labels."""
        for _ in range(n_iters):  # iterate DeepCluster steps
            for name, encoder in self.encoders_dict.items():
                z = get_latent_from_encoder(encoder, X, device=self.device)

                # --- fit KMeans ---
                kmeans = self._fit_best_kmeans_for_latent(z, cluster_range=(4,16))
                self.clusterers[name] = kmeans

                # --- soft cluster assignments ---
                distances = kmeans.transform(z)
                cluster_features = softmax(-distances, axis=1)
                cluster_features = (cluster_features - cluster_features.mean(axis=0)) / (cluster_features.std(axis=0)+1e-8)

                # --- optional lightweight refinement ---
                if refine_encoder:
                    optimizer = torch.optim.Adam(encoder.parameters(), lr=lr)
                    centers   = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32, device=self.device)
                    for _ in range(epochs_finetune):
                        # compute z fresh every step
                        z_tensor = get_latent_tensor(encoder, X, train_encoder=True, device=self.device)
                        q        = torch.softmax(-torch.cdist(z_tensor, centers), dim=1)
                        p        = torch.tensor(target_distribution(q.detach().cpu().numpy()), device=q.device)
                        loss     = F.kl_div(q.log(), p, reduction="batchmean")
                        optimizer.zero_grad()
                        loss.backward()  # no retain_graph
                        optimizer.step()
                # store/concatenate cluster features across encoders
                if self.cluster_prob_matrix is None:
                    self.cluster_prob_matrix = cluster_features
                else:
                    self.cluster_prob_matrix = np.concatenate([self.cluster_prob_matrix, cluster_features], axis=1)
    
    # remove
    def swav_soft_assign(self, X, cluster_range=(4,16), temperature=0.1):
        """Compute SWAV-style normalized soft cluster assignments for all encoders.
        Each encoder gets a moving-target distribution for pseudo-labels.
        Args:
            X (np.ndarray): input features
            cluster_range (tuple[int]): min/max clusters to try for KMeans
            temperature (float): softmax temperature to sharpen assignments
        Returns:
            dict: encoder name -> soft pseudo-label matrix (n_samples, sum_k)"""
        swav_features = {}
        for name, encoder in self.encoders_dict.items():
            z = get_latent_from_encoder(encoder, X, device=self.device)
            kmeans = self._fit_best_kmeans_for_latent(z, cluster_range)
            self.clusterers[name] = kmeans
            distances = kmeans.transform(z)  # (N, k)
            q = softmax(-distances / temperature, axis=1)
            q = q / q.sum(axis=1, keepdims=True)
            swav_features[name] = q
        return swav_features

    def deepcluster_step_swav(self, X, n_iters=1, cluster_range=(4,16), temperature=0.1, refine_encoder=False, lr=1e-3, epochs_finetune=5):
        """DeepCluster step using SWAV-style soft cluster assignments.
        Replaces hard pseudo-labels with normalized moving-target distributions.
        Args:
            X (np.ndarray): input features
            n_iters (int): number of DeepCluster iterations
            cluster_range (tuple[int]): min/max clusters for KMeans
            temperature (float): softmax temperature
            refine_encoder (bool): whether to fine-tune encoder toward clusters
            lr (float): learning rate for encoder refinement
            epochs_finetune (int): fine-tune steps per iteration"""
        if X is None or len(X) == 0:
            print("[DeepCluster] Skipping: no data provided.")
            self.cluster_prob_matrix = np.zeros((0, 0))
            return
        for _ in range(n_iters):
            for name, encoder in self.encoders_dict.items():
                # --- fit KMeans ---
                z      = get_latent_from_encoder(encoder, X, device=self.device)
                kmeans = self._fit_best_kmeans_for_latent(z, cluster_range)
                self.clusterers[name] = kmeans

                # --- SWAV-style soft assignment ---
                distances = kmeans.transform(z)
                q = softmax(-distances / temperature, axis=1)
                q = q / q.sum(axis=1, keepdims=True)  # normalize

                # store features
                if self.cluster_prob_matrix is None:
                    self.cluster_prob_matrix = q
                else:
                    self.cluster_prob_matrix = np.concatenate([self.cluster_prob_matrix, q], axis=1)

                # --- optional lightweight refinement ---
                if refine_encoder:
                    optimizer = torch.optim.Adam(encoder.parameters(), lr=lr)
                    centers   = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32, device=self.device)
                    for _ in range(epochs_finetune):
                        z_tensor = get_latent_tensor(encoder, X, train_encoder=True, device=self.device)
                        q_tensor = torch.softmax(-torch.cdist(z_tensor, centers), dim=1)
                        p_tensor = torch.tensor(target_distribution(q_tensor.detach().cpu().numpy()), device=q_tensor.device)
                        loss = F.kl_div(q_tensor.log(), p_tensor, reduction="batchmean")
                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()

def get_concat_latents(encoders_dict, X, device=device):
    """Return concatenated latent vectors from all encoders for X."""
    latents = [get_latent_tensor(encoder, X, train_encoder=False, device=device).cpu().numpy()
               for encoder in encoders_dict.values()]
    return np.concatenate(latents, axis=1)  # shape (N, sum(latent_dims))

def get_weighted_latents(encoders_dict, X, encoder_weights=None, device=device) -> np.ndarray:
    """Return concatenated latent vectors from all encoders, optionally weighted and normalized per encoder.
    - encoders_dict: dict of encoders {name: model}
    - X: input array, shape (N, T, F) or (N, F)
    - encoder_weights: dict {name: float}, defaults to 1.0 if None"""
    latents = []
    for name, encoder in encoders_dict.items():
        z = get_latent_tensor(encoder, X, train_encoder=False, device=device).cpu().numpy()
        # normalize per encoder to [0,1]
        z      = (z - z.min(axis=0, keepdims=True)) / (z.max(axis=0, keepdims=True) - z.min(axis=0, keepdims=True) + 1e-8)
        weight = 1.0 if encoder_weights is None else encoder_weights.get(name, 1.0)
        latents.append(z * weight)
    return np.concatenate(latents, axis=1)

def barlow_twins_loss(z_a: torch.Tensor, z_b: torch.Tensor, lambd: float = 0.0051, eps: float = 1e-12):
    """z_a, z_b: (B, D) - embeddings for two views, assumed zero-meaned / normalized per-dim.
    Loss = sum_i (1 - C_ii)^2 + lambda * sum_{i!=j} C_ij^2
    where C is cross-correlation matrix between z_a and z_b (B-normalized)."""
    B, D     = z_a.shape
    z_a_norm = (z_a - z_a.mean(0)) / (z_a.std(0) + eps)
    z_b_norm = (z_b - z_b.mean(0)) / (z_b.std(0) + eps)
    xcorr    = (z_a_norm.T @ z_b_norm) / B
    on_diag  = torch.diagonal(xcorr).add_(-1).pow(2).sum()
    off_diag = (xcorr - torch.diag(torch.diagonal(xcorr))).pow(2).sum()
    return on_diag + lambd * off_diag

def vicreg_loss(z_a: torch.Tensor, z_b: torch.Tensor, sim_coeff=25.0, 
                var_coeff=25.0, cov_coeff=1.0, eps=1e-4):
    """z_a, z_b : (B, D)
    sim: mean squared error between z_a and z_b
    var: hinge on std per-dim (std should be > threshold)
    cov: off-diagonal terms of covariance matrix"""
    def variance_term(z):
        std      = torch.sqrt(z.var(dim=0) + eps)
        std_loss = torch.mean(F.relu(1.0 - std))
        return std_loss

    def covariance_term(z):
        z        = z - z.mean(dim=0)
        cov      = (z.T @ z) / (B - 1)   # (D, D)
        off_diag = cov - torch.diag(torch.diagonal(cov))
        return (off_diag.pow(2).sum()) / D

    B, D     = z_a.shape
    sim_loss = F.mse_loss(z_a, z_b)
    var_loss = variance_term(z_a) + variance_term(z_b)
    cov_loss = covariance_term(z_a) + covariance_term(z_b)
    return sim_coeff * sim_loss + var_coeff * var_loss + cov_coeff * cov_loss

def train_ae_with_ssl(ae_model, X, device="cpu",
                      epochs=10, lr=1e-3, batch_size=128,
                      ssl_mode=None,      # None | 'barlow' | 'vicreg'
                      ssl_weight=1.0,     # weight applied to ssl loss
                      recon_weight=1.0,   # weight applied to reconstruction loss
                      augment_fn=None,    # function that given a torch tensor returns two views
                      print_every=10):
    """ae_model: must implement .forward(x) -> x_recon and .encode(x) -> z (torch modules)
    X: numpy array (N, T, C) or (N, D)
    augment_fn: function(X_tensor, device) -> (view1_tensor, view2_tensor)
    Returns: trained model (in-place)"""
    ae_model.to(device)
    ae_model.train()
    opt  = torch.optim.AdamW(ae_model.parameters(), lr=lr)
    N    = len(X)
    idxs = np.arange(N)

    # data to torch if needed
    X_tensor_all = torch.tensor(X, dtype=torch.float32, device=device)
    # if timeseries reshape handled by model; here we assume shape (B, T, C) or (B, D)

    for epoch in range(epochs):
        np.random.shuffle(idxs)
        for start in range(0, N, batch_size):
            batch_idx = idxs[start:start+batch_size]
            xb        = X_tensor_all[batch_idx]
            # if augment_fn provided and ssl_mode requested
            if ssl_mode is not None and augment_fn is not None:
                v1, v2 = augment_fn(xb, device)    # expected torch tensors on device
                z1 = ae_model.encode(v1).reshape(len(v1), -1)
                z2 = ae_model.encode(v2).reshape(len(v2), -1)
            else:
                # fallback: use two different random noisy versions of xb
                v1 = xb
                v2 = xb
                z1 = ae_model.encode(v1).reshape(len(v1), -1)
                z2 = ae_model.encode(v2).reshape(len(v2), -1)

            # recon: reconstruction from original (or v1)
            x_recon    = ae_model(xb)                      # assumes forward returns recon
            recon_loss = F.mse_loss(x_recon, xb)

            ssl_loss = 0.0
            if ssl_mode == "barlow":
                ssl_loss = barlow_twins_loss(z1, z2)
            elif ssl_mode == "vicreg":
                ssl_loss = vicreg_loss(z1, z2)
            elif ssl_mode is None:
                ssl_loss = 0.0
            else:
                raise ValueError("Unknown ssl_mode")

            loss = recon_weight * recon_loss + ssl_weight * ssl_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
        if (epoch + 1) % print_every == 0 or epoch == epochs-1:
            print(f"[AE+SSL] epoch {epoch+1}/{epochs} recon={recon_loss.item():.4f} ssl={float(ssl_loss):.4f}")
    ae_model.eval()
    return ae_model

def get_sliced_data(X: np.ndarray, num_slices: int, slice_idx: int) -> np.ndarray:
    """Return a station slice of X for a given encoder.
    Slices along 'pages' (axis 0)."""
    pages, _, _ = X.shape
    slice_len   = pages // num_slices
    start, end  = slice_idx * slice_len, (slice_idx + 1) * slice_len
    return X[start:end, :, :]

def get_weighted_slices(X, weights):
    """Split X along pages axis proportional to AE weights."""
    weights = np.array(weights) / np.sum(weights)
    cumsum  = np.cumsum(np.round(weights * X.shape[0])).astype(int)
    starts  = np.concatenate(([0], cumsum[:-1]))
    return [X[start:end] for start, end in zip(starts, cumsum)]

def get_weighted_slices_sqrt(X: np.ndarray, latent_dims: list[int]) -> list[np.ndarray]:
    """Split X along pages axis using sqrt(latent_dim) weighting."""
    pages   = X.shape[0]
    weights = np.sqrt(np.array(latent_dims))
    weights = weights / weights.sum()
    cum     = np.cumsum(np.round(weights * pages)).astype(int)
    starts  = np.concatenate(([0], cum[:-1]))
    return [X[start:end] for start, end in zip(starts, cum)]


In [ ]:
"Cellsup: Clustering (L+U), predictor (L), eval (test)"
# ===== params + prepare data =====
label_frac  = params["cellsup"]["label_frac"]
num_epochs  = params["cellsup"]["num_epochs"]
AE_lr       = params["cellsup"]["AE_lr"]
weight_decay= params["cellsup"]["weight_decay"]
dropout     = params["cellsup"]["dropout"]
hidden_dim  = X_train.shape[2]//2
swav_iters  = params["cellsup"]["swav_iters"]
swav_temp   = params["cellsup"]["swav_temp"]
cluster_min = params["cellsup"]["clustering"]["cluster_min"]
cluster_max = params["cellsup"]["clustering"]["cluster_max"]

n_samples = int(label_frac * len(X_train))
indices   = np.random.permutation(len(X_train))
X_L = X_train[indices[:n_samples]]
y_L = y_train_scaled[indices[:n_samples]]
X_U = X_train[indices[n_samples:]]  # optional unlabeled
y_U = y_train_scaled[indices[n_samples:]]  # optional if needed for semi-supervised
X_all     = np.concatenate([X_L, X_U], axis=0)
input_dim = X_all.shape[1] * X_all.shape[2] if X_all.ndim == 3 else X_all.shape[1]
X_tensor  = torch.tensor(X_train, dtype=torch.float32, device=device).reshape(len(X_train), -1)

# ===== pretrain section =====
# Pretraining step: each encoder learns X > z > X_recon. After pretraining, encoder is frozen for downstream tasks

# ===== pretrain AE variants =====
ae_encoders = {}
encoders_dims_list = params["cellsup"]["encoders_dims_list"]
# for i, latent_dim in enumerate(encoders_dims_list):
#     ae_model = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
#     ae_model = train_ae_with_bootstraps(ae_model, X_train, num_epochs=num_epochs, lr=AE_lr,
#                                         sample_frac=0.8, weight_decay=weight_decay, device=device)
#     ae_encoders[f"AE_{latent_dim}"] = ae_model

"even slicing"
# num_slices = len(encoders_dims_list)
# for i, latent_dim in enumerate(encoders_dims_list):
#     X_train_slice   = get_sliced_data(X_train, num_slices, i)
#     input_dim_slice = X_train_slice.shape[1] * X_train_slice.shape[2]
#     ae_model = ae.FlexibleAutoencoder(layer_dims=[input_dim_slice, 64, latent_dim], pred_dim=0).to(device)
#     ae_model = train_ae_with_bootstraps(ae_model, X_train_slice, num_epochs=num_epochs,lr=AE_lr,
#                                         sample_frac=0.8, weight_decay=weight_decay, device=device)
#     ae_encoders[f"AE_slice{i}_dim{latent_dim}"] = ae_model

"weighted slicing"
weights  = np.array(encoders_dims_list) / np.sum(encoders_dims_list)
# X_slices = get_weighted_slices(X_train, weights)
X_slices = get_weighted_slices_sqrt(X_train, encoders_dims_list)
for i, (latent_dim, X_train_slice) in enumerate(zip(encoders_dims_list, X_slices)):
    input_dim_slice = X_train_slice.shape[1] * X_train_slice.shape[2]
    ae_model        = ae.FlexibleAutoencoder(layer_dims=[input_dim_slice, hidden_dim, latent_dim], pred_dim=0).to(device)
    ae_model = train_ae_with_bootstraps(ae_model,X_train_slice,num_epochs=num_epochs,lr=AE_lr,
                                        sample_frac=0.8,weight_decay=weight_decay,device=device)
    ae_encoders[f"AE_slice{i}_dim{latent_dim}"] = ae_model

# ===== pretrain Denoising AE =====
# denoise_ae    = ae.DenoisingAE(input_size=input_dim, hidden_dims=[64,16], latent_dim=8,
#                                dropout_prob=dropout, noise_std=0.1).to(device)
# optimizer_dae = torch.optim.AdamW(denoise_ae.parameters(), lr=AE_lr, weight_decay=weight_decay)
# for epoch in range(num_epochs):
#     optimizer_dae.zero_grad()
#     X_recon = denoise_ae(X_tensor)
#     loss    = F.mse_loss(X_recon, X_tensor)
#     loss.backward()
#     optimizer_dae.step()

# ===== assemble encoders =====
encoders_dict = {**ae_encoders,
                #  "denoiseAE": denoise_ae,
                 }

# Add a suphead for each encoder
# sup_head_rmse = {}
# for name, encoder in encoders_dict.items():
#     rmse = train_sup_head_per_encoder(encoder, X_L, y_L, X_test, y_test_scaled, dropout,
#                                       all_encoders=encoders_dict, reg_ortho=1e-3,   # tune this
#                                       train_encoder=True, device=device, epochs=num_epochs)
#     sup_head_rmse[name] = rmse
#     print(f"{name}: RMSE = {rmse:.4f}")
# ===== Train sup-heads (optionally finetune encoders) =====
sup_head_rmse = train_sup_heads_joint(encoders_dict, X_L, y_L, X_test, y_test_scaled,
                                      hidden_sizes=[64,32], lr=AE_lr, epochs=num_epochs,
                                      device=device, train_encoders=True, reg_ortho=0e-3)
for name, rmse in sup_head_rmse.items():
    print(f"{name}: RMSE = {rmse:.4f}")

# xxxxxxxxxx Per-encoder evaluation xxxxxxxxxx
print("Per-encoder CatBoost RMSE:")
for name, encoder in encoders_dict.items():
    z_train = get_latent_tensor(encoder, X_L, train_encoder=False, device=device).cpu().numpy()
    z_test  = get_latent_tensor(encoder, X_test, train_encoder=False, device=device).cpu().numpy()
    _, rmse = train_and_eval_catboost(z_train, y_L, z_test, y_test_scaled)
    print(f"   {name}: {rmse:.4f}")

# ===== Encoder weights =====
weight_encoding_method = "inverse_rmse"  # "uniform", "inverse_rmse", "softmax"
encoder_weights = assign_encoder_weights(encoders_dict, sup_head_rmse, weight_encoding_method)

# ===== ensemble clustering =====
n_clusters = 8
ensemble_clusters = Cellsup(encoders_dict=encoders_dict, n_clusters=n_clusters,
                            device=device, cluster_assignment="soft", cluster_metric="ch")

# """§0 BASELINE: Pure supervised on latents (no clustering, no pseudo-labels)"""
# z_train_concat = get_weighted_latents(encoders_dict, X_L, encoder_weights, device=device)
# z_test_concat  = get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)
# _, rmse        = train_and_eval_catboost(z_train_concat, y_L, z_test_concat, y_test_scaled)
# print(f"  >> §0 latent (no cluster z>y) CatBoost RMSE: {rmse:.4f}")
# linreg_loss0, catboost_loss0, unsupervised_rmse0, rf_rmse0 = \
#     Preds.evaluate_models_on_dataset(z_train_concat, y_L, z_test_concat, y_test_scaled)

"""§1 Clustering (clusters > pseudo-labels > RMSE)"""
ensemble_clusters.encoders_dict = encoders_dict
print("Encoders used for pseudo-labels:", list(ensemble_clusters.encoders_dict.keys()))
X_all_aug    = np.concatenate([X_L, X_U], axis=0)
z_all_concat = get_weighted_latents(encoders_dict, X_all_aug, encoder_weights, device=device)
z_test_concat= get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)

ensemble_clusters.fit_kmeans_on_encoder_latents(X_L, encoder_weights=encoder_weights, cluster_range=(cluster_min, cluster_max))
y_U_pseudo   = ensemble_clusters.assign_pseudo_labels(X_L, y_L, X_U, confidence_thresh=0)
y_all_aug    = np.concatenate([y_L, y_U_pseudo], axis=0)
_, rmse      = train_and_eval_catboost(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)
print(f"  >> §1 Semi-supervised latent+cluster CatBoost RMSE: {rmse:.4f}")
cellsup_losses = Preds.evaluate_models_on_dataset(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

"""§2 DeepCluster (z > clusters > rmse)"""
ensemble_clusters.cluster_prob_matrix = None
multiview_bool = True
# if len(encoders_dict) == 1:
#     multiview_bool = False

if multiview_bool:
    X_U_torch    = torch.tensor(X_U, dtype=torch.float32, device=device)
    X_U_view1, X_U_view2 = make_two_views_augmentation(X_U_torch, device, scale=0.1)
    X_U_aug      = torch.cat([X_U_view1, X_U_view2], dim=0).cpu().numpy()
    ensemble_clusters.deepcluster_step_swav(X_U_aug, n_iters=swav_iters, cluster_range=(4, 16),
                                            temperature=swav_temp, refine_encoder=False)
else:
    ensemble_clusters.deepcluster_step_swav(X_U, n_iters=swav_iters, cluster_range=(4,16),
                                            temperature=swav_temp, refine_encoder=False)

swav_feats_U          = ensemble_clusters.cluster_prob_matrix  # now shape (N_unlabeled, sum_k)
encoder_cluster_sizes = [ensemble_clusters.clusterers[name].n_clusters for name in encoders_dict]
start = 0
per_encoder_means = []
for k in encoder_cluster_sizes:
    per_encoder_means.append(np.mean(swav_feats_U[:, start:start+k], axis=1, keepdims=True))
    start += k
y_dim      = y_L.shape[1]
y_U_pseudo = np.mean(np.concatenate(per_encoder_means, axis=1), axis=1, keepdims=True)  # (N_unlabeled, 1)
y_U_pseudo = y_U_pseudo[:len(X_U)]  

y_U_pseudo_full = np.tile(y_U_pseudo, (1, y_dim))  # (N_unlabeled, y_dim)
X_all_aug       = np.concatenate([X_L, X_U], axis=0)
y_all_aug       = np.concatenate([y_L, y_U_pseudo_full], axis=0)
z_all_concat    = get_weighted_latents(encoders_dict, X_all_aug, encoder_weights, device=device)
z_test_concat   = get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)

_, rmse = train_and_eval_catboost(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)
print(f"  >> §2 DeepCluster latent+cluster CatBoost RMSE: {rmse:.4f}")
swav_losses = Preds.evaluate_models_on_dataset(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

# """§3 Barlow Twins (SSL consistency regularizer on unlabeled data)"""
# print(">> Running §3 Barlow Twins consistency step")
# z_view1_concat = get_weighted_latents(encoders_dict, X_U_view1.cpu().numpy(), encoder_weights, device=device)
# z_view2_concat = get_weighted_latents(encoders_dict, X_U_view2.cpu().numpy(), encoder_weights, device=device)

# # compute BT loss (as regularization indicator, not for training)
# loss_BT = barlow_twins_loss(
#     torch.tensor(z_view1_concat, device=device, dtype=torch.float32),
#     torch.tensor(z_view2_concat, device=device, dtype=torch.float32),)
# print(f"  >> §3 Barlow Twins unsupervised loss: {loss_BT.item():.4f}")

# # optionally, use BT consistency as pseudo-supervision
# z_all_concat  = get_weighted_latents(encoders_dict, np.concatenate([X_L, X_U]), encoder_weights, device=device)
# z_test_concat = get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)

# # make pseudo-targets = avg 2 BT views’ means (simple consistency trick)
# y_U_pseudo      = (z_view1_concat.mean(axis=1, keepdims=True) + z_view2_concat.mean(axis=1, keepdims=True))/2
# y_U_pseudo_full = np.tile(y_U_pseudo, (1, y_L.shape[1]))
# y_all_aug       = np.concatenate([y_L, y_U_pseudo_full], axis=0)

# _, rmse = train_and_eval_catboost(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)
# print(f"  >> §3 Barlow Twins latent+consistency CatBoost RMSE: {rmse:.4f}")
# linreg_loss3, catboost_loss3, unsupervised_rmse3, rf_rmse3 = \
#     Preds.evaluate_models_on_dataset(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

print(f"Results for dataset: {desired_dataset}, {label_frac=}")
print("    RMSE       | LinReg | CatBoost | Cluster | RForest")
# print(f"& Z (concat)   & {linreg_loss0:.4f} & {catboost_loss0:.4f}   & {unsupervised_rmse0:.4f}  & {rf_rmse0:.4f} \\\\")
print(f"& Z (pseudo)   & {cellsup_losses[0]:.4f} & {cellsup_losses[1]:.4f}   & {cellsup_losses[2]:.4f} \\\\")
print(f"& Z (swav)     & {swav_losses[0]:.4f} & {swav_losses[1]:.4f}   & {swav_losses[2]:.4f}  \\\\")
# print(f"& Z (Barlow)   & {linreg_loss3:.4f} & {catboost_loss3:.4f}   & {unsupervised_rmse3:.4f} & {rf_rmse3:.4f} \\\\")

print(f" & \\val{{{cellsup_losses[0]:.3f}}}{{}}"
      f" & \\val{{{cellsup_losses[1]:.3f}}}{{}}"
      f" & \\val{{{cellsup_losses[2]:.3f}}}{{}} \\\\")


In [ ]:
"KAN net"
if params["run_console"]["kan"] == True:
    from kan import KAN

    # X_train_flat = X_train.reshape(X_train.shape[0], -1)
    # X_test_flat  = X_test.reshape(X_test.shape[0], -1)
    X_train_mean = X_train.mean(axis=1)
    X_test_mean  = X_test.mean(axis=1)

    X_train_t = torch.tensor(X_train_mean, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
    X_test_t  = torch.tensor(X_test_mean, dtype=torch.float32)
    y_test_t  = torch.tensor(y_test_scaled, dtype=torch.float32)

    dataset = {
        'train_input': X_train_t,
        'train_label': y_train_t,
        'test_input': X_test_t,
        'test_label': y_test_t,
        'train_ratio': 0.8}

    model = KAN(width=[X_train_t.shape[1], 128, y_train_t.shape[1]], grid=5, k=3)

    print("Starting training...")
    results = model.fit(
        dataset, 
        opt="LBFGS",
        steps=5,
        lamb=0.01)

    print("\nTraining Complete.")
    print(f"Final training loss: {results['train_loss'][-1]:.4e}")
    print(f"Final test loss: {results['test_loss'][-1]:.4e}")
    with torch.no_grad():
        y_pred_t = model(X_test_t)
    y_pred = y_pred_t.cpu().numpy()

    rmse = root_mean_squared_error(y_test_t.cpu().numpy(), y_pred)
    print(f"Test RMSE: {rmse:.4f}")

    X_train_mean = X_train.mean(axis=1)
    X_test_mean  = X_test.mean(axis=1)

    X_train_t = torch.tensor(X_train_mean, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
    X_test_t  = torch.tensor(X_test_mean, dtype=torch.float32)
    y_test_t  = torch.tensor(y_test_scaled, dtype=torch.float32)

    model = KAN(width=[X_train_t.shape[1], 64, 32, y_train_t.shape[1]], grid=5, k=3)

    results = model.fit(
        {'train_input': X_train_t, 'train_label': y_train_t,
        'test_input': X_test_t,  'test_label': y_test_t,
        'train_ratio': 0.8},
        opt="LBFGS", steps=30, lamb=0.01, update_grid=True)

    with torch.no_grad():
        y_pred_t = model(X_test_t)
    y_pred = y_pred_t.cpu().numpy()

    rmse = root_mean_squared_error(y_test_t.cpu().numpy(), y_pred)
    print(f"[KAN] Test RMSE: {rmse:.4f}")
    model.plot()


In [ ]:
"BARLOW + CNN"

class CnnAutoencoder(nn.Module):
    def __init__(self, n_features, n_timesteps, latent_dim):
        super().__init__()
        self.n_timesteps = n_timesteps
        self.n_features  = n_features
        
        # --- ENCODER (CNN) ---
        self.conv1 = nn.Conv1d(n_features, CHANNELS_1, KERNEL_SIZE)
        self.pool1 = nn.MaxPool1d(POOL_KERNEL)
        self.conv2 = nn.Conv1d(CHANNELS_1, CHANNELS_2, KERNEL_SIZE)
        self.pool2 = nn.MaxPool1d(POOL_KERNEL)
        
        # Calculate flat_size dynamically (using dummy data for clean calculation)
        with torch.no_grad():
            dummy_input = torch.zeros(1, n_timesteps, n_features).permute(0, 2, 1)
            x           = self.pool1(F.relu(self.conv1(dummy_input)))
            x           = self.pool2(F.relu(self.conv2(x)))
            self.flat_size = x.numel() # automatically calculate size

        self.enc_linear = nn.Linear(self.flat_size, latent_dim) # encode
        self.dec_linear = nn.Linear(latent_dim, self.flat_size) # decode
        
        # Unpooling and Deconvolution require remembering the output shape of the *encoder's last pool*
        self.dec_flat_time     = x.shape[2] # Save the time dimension length here
        self.dec_flat_channels = CHANNELS_2
        
        # Use ConvTranspose1d to upsample (reverse of Conv1d)
        self.deconv2 = nn.ConvTranspose1d(CHANNELS_2, CHANNELS_1, KERNEL_SIZE)
        self.deconv1 = nn.ConvTranspose1d(CHANNELS_1, n_features, KERNEL_SIZE)#, output_padding=1)

    def encode(self, x):
        # x shape: (B, T, F) -> (B, F, T)
        x = x.permute(0, 2, 1)
        
        self.shape_after_pool2 = (x.shape[0], self.dec_flat_channels, self.dec_flat_time)
        
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        
        # Flatten and produce latent vector Z
        x = x.view(-1, self.flat_size)
        z = self.enc_linear(x)
        return z

    def decode(self, z):
            x = F.relu(self.dec_linear(z))
            x = x.view(self.shape_after_pool2) # (B, 128, T_flat)

            # First upsample and deconv
            x = F.interpolate(x, scale_factor=POOL_KERNEL, mode='nearest')
            x = F.relu(self.deconv2(x)) # Output size is L_in + K - 1 (since stride=1, padding=0)

            # Second upsample and deconv (Target length: self.n_timesteps)
            x = F.interpolate(x, scale_factor=POOL_KERNEL, mode='nearest')
            
            # Calculate the required output_padding for the final layer
            # Desired_Length = self.n_timesteps
            # L_in (current size) = x.shape[2]
            # L_out = L_in + K - 1 + output_padding (since stride=1, padding=0)
            # Required_output_padding = Desired_Length - (L_in + K - 1)
            
            current_size = x.shape[2]
            required_padding = self.n_timesteps - (current_size + KERNEL_SIZE - 1)
            
            # Ensure padding is non-negative and integer
            required_padding = max(0, int(required_padding))
            
            # x = self.deconv1(x, output_padding=required_padding) # FIX: Apply padding here
            x = self.deconv1(x)

            diff = self.n_timesteps - x.shape[2]
            if diff > 0:
                x = F.pad(x, (0, diff))
            elif diff < 0:
                x = x[:, :, :self.n_timesteps]
            return x.permute(0, 2, 1)
    
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)

SSL_LAMBDA  = params["barlow"]["cnn"]["SSL_LAMBDA"] # value from BT paper
SSL_WEIGHT  = params["barlow"]["cnn"]["SSL_WEIGHT"] # Weight for recon vs. SSL
CHANNELS_1  = params["barlow"]["cnn"]["CHANNELS_1"]
CHANNELS_2  = params["barlow"]["cnn"]["CHANNELS_2"]
KERNEL_SIZE = params["barlow"]["cnn"]["KERNEL_SIZE"]
POOL_KERNEL = params["barlow"]["cnn"]["POOL_KERNEL"]
latent_dim  = params["barlow"]["cnn"]["latent_dim"]
augment_const_cnn = params["barlow"]["cnn"]["augment_const"]


n_timesteps = X_train.shape[1]
n_features  = X_train.shape[2]

# Replace the old AE model initialization
cnn_ae    = CnnAutoencoder(n_features, n_timesteps, latent_dim).to(device)
optimizer = torch.optim.AdamW(cnn_ae.parameters(), lr=AE_lr)
# X_tensor must now be the 3D sequence data (B, T, F)
X_tensor  = torch.tensor(X_train, dtype=torch.float32, device=device) # NO .reshape(len(X_train), -1)

# ===== CNN ENCODER + Barlow Twins SSL =====
cnn_ae.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()

    v1, v2 = make_two_views_augmentation(X_tensor, device, augment_const_cnn)
    z1 = cnn_ae.encode(v1) # Z1 shape: (B, latent_dim)
    z2 = cnn_ae.encode(v2) # Z2 shape: (B, latent_dim)
    
    # 3. Barlow Twins Loss
    B, D     = z1.shape
    z1_norm  = (z1 - z1.mean(0)) / (z1.std(0) + 1e-12)
    z2_norm  = (z2 - z2.mean(0)) / (z2.std(0) + 1e-12)
    xcorr    = (z1_norm.T @ z2_norm) / B
    on_diag  = torch.diagonal(xcorr).add_(-1).pow(2).sum()
    off_diag = (xcorr - torch.diag(torch.diagonal(xcorr))).pow(2).sum()
    ssl_loss = on_diag + SSL_LAMBDA * off_diag

    # 4. Final Loss
    X_recon    = cnn_ae(X_tensor) 
    recon_loss = F.mse_loss(X_recon, X_tensor) # Now shapes match: (B, T, F) vs (B, T, F)
    loss_total = SSL_WEIGHT * ssl_loss + recon_loss
    loss_total.backward()
    optimizer.step()
    if (epoch + 1) % 3 == 0 or epoch == num_epochs-1:
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {loss_total.item():.4f}")
cnn_ae.eval()

# ===== GET LATENT VECTORS FOR CATBOOST =====
X_L_tensor    = torch.tensor(X_L, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)

with torch.no_grad():
    z_train = cnn_ae.encode(X_L_tensor).cpu().numpy()
    z_test  = cnn_ae.encode(X_test_tensor).cpu().numpy()

# ===== EVALUATE =====
barlow_cnn_losses = Preds.evaluate_models_on_dataset(z_train, y_L, z_test, y_test_scaled)
print(f"Results:\nLinReg: {barlow_cnn_losses[0]:.4f} | CatBoost: {barlow_cnn_losses[1]:.4f} | RForest: {barlow_cnn_losses[2]:.4f}")
print(f" & \\val{{{barlow_cnn_losses[0]:.3f}}}{{}}"
      f" & \\val{{{barlow_cnn_losses[1]:.3f}}}{{}}"
      f" & \\val{{{barlow_cnn_losses[2]:.3f}}}{{}} \\\\")


In [ ]:
"BARLOW + AE"
latent_dim   = params["barlow"]["ae"]["latent_dim"]
ssl_weight   = params["barlow"]["ae"]["ssl_weight"]
augment_const_ae= params["barlow"]["ae"]["augment_const"]

input_dim = X_train.shape[1] * X_train.shape[2] if X_train.ndim == 3 else X_train.shape[1]
X_tensor  = torch.tensor(X_train, dtype=torch.float32, device=device).reshape(len(X_train), -1)

# ae_model  = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
ae_model  = ae.FlexibleAutoencoder(layer_dims=[input_dim, input_dim//4, input_dim//12, latent_dim], pred_dim=0).to(device)
optimizer = torch.optim.AdamW(ae_model.parameters(), lr=AE_lr)

# ===== TRAIN AE WITH Barlow Twins SSL =====
ae_model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    X_recon    = ae_model(X_tensor)
    recon_loss = F.mse_loss(X_recon, X_tensor)
    X3d     = X_train if X_train.ndim == 3 else X_train.reshape(len(X_train), -1, 1)
    X3d     = torch.tensor(X3d, dtype=torch.float32, device=device)
    v1, v2  = make_two_views_augmentation(X3d, device, augment_const_ae)
    v1_flat = v1.reshape(len(v1), -1)
    v2_flat = v2.reshape(len(v2), -1)
    assert v1_flat.shape[1] == input_dim, f"Expected {input_dim}, got {v1_flat.shape[1]}"

    z1 = ae_model.encode(v1_flat)
    z2 = ae_model.encode(v2_flat)

    # Barlow Twins
    B, D     = z1.shape
    z1_norm  = (z1 - z1.mean(0)) / (z1.std(0) + 1e-12)
    z2_norm  = (z2 - z2.mean(0)) / (z2.std(0) + 1e-12)
    xcorr    = (z1_norm.T @ z2_norm) / B
    on_diag  = torch.diagonal(xcorr).add_(-1).pow(2).sum()
    off_diag = (xcorr - torch.diag(torch.diagonal(xcorr))).pow(2).sum()
    ssl_loss = on_diag + SSL_LAMBDA * off_diag
    loss     = recon_loss + ssl_weight * ssl_loss
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}/{num_epochs} - Recon: {recon_loss.item():.4f} SSL: {ssl_loss.item():.4f}")

ae_model.eval()
z_train = get_latent_from_encoder(ae_model, X_L, device=device)
z_test  = get_latent_from_encoder(ae_model, X_test, device=device)

# ===== TRAIN CATBOOST & EVALUATE =====
_, rmse = train_and_eval_catboost(z_train, y_L, z_test, y_test_scaled)
print(f"AE+Barlow latent CatBoost RMSE: {rmse:.4f}")

barlow_ae_losses = Preds.evaluate_models_on_dataset(z_train, y_L, z_test, y_test_scaled)
print(f"Results:\nLinReg: {barlow_ae_losses[0]:.4f} | CatBoost: {barlow_ae_losses[1]:.4f} | RForest: {barlow_ae_losses[2]:.4f}")
print(f" & \\val{{{barlow_ae_losses[0]:.3f}}}{{}}"
      f" & \\val{{{barlow_ae_losses[1]:.3f}}}{{}}"
      f" & \\val{{{barlow_ae_losses[2]:.3f}}}{{}} \\\\")


In [ ]:
"BARLOW + AE (SSL)"

# ==== BARLOW TWINS + AE (Semi-Supervised) ====
latent_dim = 32
ssl_weight = 1.0
sup_weight = 0.5  # weight for supervised fine-tuning
SSL_LAMBDA = 0.005

# ===== Tensors =====
# Use only labeled subset
X_L_tensor = torch.tensor(X_L, dtype=torch.float32, device=device).reshape(len(X_L), -1)
y_L_tensor = torch.tensor(y_L, dtype=torch.float32, device=device)
if y_L_tensor.ndim == 1:
    y_L_tensor = y_L_tensor.view(-1, 1)

# ===== Model =====
ae_model = ae.FlexibleAutoencoder(
    layer_dims=[input_dim, input_dim//4, input_dim//12, latent_dim],pred_dim=1).to(device)
optimizer = torch.optim.AdamW(ae_model.parameters(), lr=AE_lr)

# 1️⃣ Stage 1 — Self-Supervised Pretraining (Barlow Twins)
print("\n=== Stage 1: Self-Supervised Pretraining (Barlow Twins) ===")
ae_model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()

    # reconstruction loss
    X_recon = ae_model(X_tensor, mode="reconstruct")
    recon_loss = F.mse_loss(X_recon, X_tensor)

    # augmentations for Barlow Twins
    X3d     = X_train if X_train.ndim == 3 else X_train.reshape(len(X_train), -1, 1)
    X3d     = torch.tensor(X3d, dtype=torch.float32, device=device)
    v1, v2  = make_two_views_augmentation(X3d, device, 0.1)
    v1_flat = v1.reshape(len(v1), -1)
    v2_flat = v2.reshape(len(v2), -1)

    # encodings
    z1 = ae_model.encode(v1_flat)
    z2 = ae_model.encode(v2_flat)

    # Barlow Twins loss
    B, D     = z1.shape
    z1_norm  = (z1 - z1.mean(0)) / (z1.std(0) + 1e-12)
    z2_norm  = (z2 - z2.mean(0)) / (z2.std(0) + 1e-12)
    xcorr    = (z1_norm.T @ z2_norm) / B
    on_diag  = torch.diagonal(xcorr).add_(-1).pow(2).sum()
    off_diag = (xcorr - torch.diag(torch.diagonal(xcorr))).pow(2).sum()
    ssl_loss = on_diag + SSL_LAMBDA * off_diag

    # combined SSL + AE loss
    loss = recon_loss + ssl_weight * ssl_loss
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}/{num_epochs} - Recon: {recon_loss.item():.4f} SSL: {ssl_loss.item():.4f}")

# 2️⃣ Stage 2 — Supervised Fine-Tuning (on labeled subset)
print("\n=== Stage 2: Supervised Fine-Tuning ===")
for epoch in range(max(5, num_epochs // 2)):
    optimizer.zero_grad()
    y_pred = ae_model(X_L_tensor, mode="predict")  # use forward(mode="predict")
    sup_loss = F.mse_loss(y_pred, y_L_tensor)      # shapes now match
    sup_loss.backward()
    optimizer.step()
    print(f"Fine-tune {epoch+1} - Supervised Loss: {sup_loss.item():.4f}")

# 3️⃣ Evaluation
ae_model.eval()
z_train = get_latent_from_encoder(ae_model, X_L, device=device)
z_test  = get_latent_from_encoder(ae_model, X_test, device=device)

barlow_ae_ssl_loss = Preds.evaluate_models_on_dataset(
    z_train, y_L, z_test, y_test_scaled)

print(f"\n=== Results ===\nLinReg: {barlow_ae_ssl_loss[0]:.4f} | CatBoost: {barlow_ae_ssl_loss[1]:.4f} | "
      f"Cluster: {barlow_ae_ssl_loss[2]:.4f} | RForest: {barlow_ae_ssl_loss[2]:.4f}")
print(f" & \\val{{{barlow_ae_ssl_loss[0]:.3f}}}{{}}"
      f" & \\val{{{barlow_ae_ssl_loss[1]:.3f}}}{{}}"
      f" & \\val{{{barlow_ae_ssl_loss[2]:.3f}}}{{}} \\\\")


In [ ]:
"Direct pred. (X > y) on dataset with missing labels"
# Train on X_L predict on X_test (no mention of X_U)

# ====== X mean ======
X_L_2d     = X_L.mean(axis=1).astype(np.float32)
X_test_2d  = X_test.mean(axis=1).astype(np.float32)
mean_losses= Preds.evaluate_models_on_dataset(X_L_2d, y_L, X_test_2d, y_test_scaled)
_, _, nn_rmse = Preds.predict_mlp_multioutput(X_L_2d, y_L, X_test_2d, y_test_scaled,
                                              hidden_layer_sizes=(128, 64), max_iter=500)
print(f"Results on raw features ({int(label_frac*100)}% labeled):")
print("    RMSE   | LinReg | CatBoost | Cluster | RForest | NN")
print(f"& X (mean) & {mean_losses[0]:.4f} & {mean_losses[1]:.4f}   & {mean_losses[2]:.4f} & {nn_rmse:.4f} \\\\")

print(f" & \\val{{{mean_losses[0]:.3f}}}{{}}"
      f" & \\val{{{mean_losses[1]:.3f}}}{{}}"
      f" & \\val{{{mean_losses[2]:.3f}}}{{}} \\\\")

# # ====== X random ======
# n_train, rows, _ = X_train.shape
# n_test     = X_test.shape[0]
# train_idx  = np.random.randint(0, rows, size=n_train)
# test_idx   = np.random.randint(0, rows, size=n_test)
# X_train_2d = X_train[np.arange(n_train), train_idx, :].astype(np.float32)
# X_test_2d  = X_test[np.arange(n_test), test_idx, :].astype(np.float32)
# linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse = \
#     Preds.evaluate_models_on_dataset(X_train_2d, y_train_scaled, X_test_2d, y_test_scaled)
# print(f"& random(X)& {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f} & - \\\\")

# print(f" & \\val{{{linreg_loss:.3f}}}{{}}"
#       f" & \\val{{{catboost_loss:.3f}}}{{}}"
#       f" & \\val{{{rf_rmse:.3f}}}{{}} \\\\")

# # ====== flatten ======
# X_train_flat = X_train.reshape(X_train.shape[0], -1)
# X_test_flat  = X_test.reshape(X_test.shape[0], -1)
# n_samples = int(label_frac * X_train_flat.shape[0])
# X_flat_L  = X_train_flat[:n_samples]
# y_flat_L  = y_train_scaled[:n_samples]

# X_train_t = torch.tensor(X_flat_L, dtype=torch.float32)
# y_train_t = torch.tensor(y_flat_L, dtype=torch.float32)
# X_test_t  = torch.tensor(X_test_flat, dtype=torch.float32)
# y_test_t  = torch.tensor(y_test_scaled, dtype=torch.float32)

# linreg_loss, catboost_loss, unsupervised_rmse, rf_rmse = \
#     Preds.evaluate_models_on_dataset(X_train_flat, y_train_scaled, X_test_flat, y_test_scaled)
# print(f"& flatten(X)& {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f} & - \\\\")
# print(f" & \\val{{{linreg_loss:.3f}}}{{}}"
#       f" & \\val{{{catboost_loss:.3f}}}{{}}"
#       f" & \\val{{{rf_rmse:.3f}}}{{}} \\\\")

# # ====== define FNN ======
# input_dim  = X_train_t.shape[1]
# output_dim = y_train_t.shape[1] if y_train_t.ndim > 1 else 1
# predictor_hidden_dims = [128, 64]
# predictor_lr      = 1e-3
# predictor_epochs  = 500
# predictor_dropout = 0.0
# nn_predictor = MLPHead(input_dim  = input_dim,
#                        output_dim = output_dim,
#                        hidden_sizes = predictor_hidden_dims,
#                        lr = predictor_lr,
#                        epochs  = predictor_epochs,
#                        dropout = predictor_dropout,
#                        early_stop_patience = 30,
#                        device  = device)
# nn_predictor.train(X_train_t, y_train_t, X_test_t, y_test_t)
# nn_rmse = nn_predictor.evaluate(X_test_t, y_test_t)
# print(f"NN RMSE (MLPHead, label_frac={label_frac}): {nn_rmse:.4f}")


In [ ]:
"Saving to file"
RESULTS_FILE       = "results/results_numbers.json"
LATEX_RESULTS_FILE = "results/latex_results.txt"

def load_json_safely(path: str) -> dict:
    """Load JSON file or return empty dict if missing/empty/invalid."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return {}
    try:
        return json.load(open(path))
    except json.JSONDecodeError:
        return {}

def log_result_to_json(method: str, values: list[float]):
    """Append one run's list/tuple of metrics to results file."""
    data = load_json_safely(RESULTS_FILE)
    data.setdefault(method, []).append(list(values))
    json.dump(data, open(RESULTS_FILE, "w"), indent=2)

log_result_to_json("mean(X)", mean_losses)
log_result_to_json("TS2Vec", ts2vec_losses)
log_result_to_json("TimeVAE", timevae_losses)
log_result_to_json("Moment", moment_losses)
log_result_to_json("Cellsup", cellsup_losses)
log_result_to_json("BarlowCNN", barlow_cnn_losses)
log_result_to_json("BarlowAE_SSL", barlow_ae_ssl_loss)
log_result_to_json("BarlowAE", barlow_ae_losses)


# ---- Read JSON ----
data = load_json_safely(RESULTS_FILE)

# Check if all methods have a multiple-of-5 number of runs
if data and all(len(runs) % 5 == 0 for runs in data.values()):  # change back to 5 when ready
    with open(LATEX_RESULTS_FILE, "a") as f:  # <<< APPEND MODE
        timestamp = datetime.now().strftime("%H:%M")
        f.write(f"---- {timestamp} ----\n")
        for method, runs in data.items():
            arr = np.array(runs[-5:])  # only last 5 runs
            means, stds = arr.mean(axis=0), arr.std(axis=0)
            line = f"{method} & " + " & ".join(
                f"\\val{{{m:.3f}}}{{{std:.3f}}}" for m, std in zip(means, stds)) + "\n"
            f.write(line)
    print(f"Appended avg/std to {LATEX_RESULTS_FILE}")
else:
    print("Not enough runs yet (need multiple of 5 per method).")